# Hashing & Hash Tables: Zero to Hero

**NB-03 in the [DSA: Zero to Hero](README.md) series.**

From "what must a hash function actually guarantee" to constructing the input that turns a
production dictionary quadratic — with both table designs built from scratch, every invariant
asserted after every operation, and every cost measured against the formula that predicts it.

***

## Why this notebook is different

Three things here are usually asserted and are measured instead:

- **"Hash tables are O(1)."** They are *expected* O(1), and §3 constructs the input that makes
  CPython's `dict` **quadratic** — thousands of times slower at n = 16,000, and the gap grows 4×
  per doubling. The keys are ordinary integers. No trickery, about six lines of code.
- **"Use a good hash function."** §1.1 measures four of them on the same 20,000 keys. The two bad
  ones leave **86% and 99% of buckets empty**; the two good ones land within a few percent of the
  uniform prediction. "Good" turns out to be a measurable property, not a vibe.
- **Load factor is where the two table designs diverge.** §1.5 measures both against their
  textbook formulas. Chaining degrades *linearly* in α and matches theory to within 1% even at
  α = 8. Linear probing degrades *quadratically* and costs **about 190× more probes than
  chaining** at α = 0.95.

And one finding that came out of writing it: **Python and Java are each wide open to the attack
the other resists.** Java's `String.hashCode` is a fixed function, so §3 constructs 2ᵏ strings
with an identical hash in one line — and Python's randomised SipHash makes those same strings
land in 8 different buckets, costing it nothing. Meanwhile Python's `hash(i) == i` for integers
leaves it defenceless where Java's `HashMap` shrugs the attack off by turning long buckets into
trees. Two different defences, each with the other's hole.

***

## Contents

**Part 1 — Theory from zero**
1. What a hash function must guarantee, measured
2. Collisions are not bad luck, they are arithmetic
3. **Chaining**, built from scratch
4. **Open addressing**, built from scratch — and the delete bug
5. **Load factor**, measured against the formulas
6. Java: the `hashCode`/`equals` contract

**Part 2 — Worked problems** · **Part 3 — The signature difficulty: adversarial input**
**Part 4 — Tough questions** · **Part 5 — Practice** · **Part 6 — Reading**

***

## In one paragraph

A hash table trades **worst-case guarantees for average-case speed**. It computes an integer from
a key, reduces it modulo the table size, and stores the entry there — so a lookup touches one
slot instead of searching. That is $\Theta(1)$ *expected*, and the entire subject is the gap
between "expected" and "guaranteed". Collisions are unavoidable (§1.2: with 4,096 buckets the
first collision arrives after about 80 keys, not 4,096), so every table needs a collision
strategy: **chaining** puts a list in each bucket, **open addressing** walks to the next free
slot. Both are $\Theta(1)$ expected and $\Theta(n)$ worst case, and they degrade very differently
as the table fills — which is what the **load factor** measures and §1.5 quantifies. The failure
modes are the interesting part: a bad hash function concentrates keys (§1.1), a delete that
clears a slot instead of marking it silently loses unrelated keys (§1.4), a mutable key becomes
unreachable while still consuming memory (§1.6), and an adversary who can predict your hash turns
your $\Theta(1)$ into $\Theta(n)$ on purpose (§3). Hash tables are the most-used data structure
in software and the one whose stated complexity is most often wrong.

**Prerequisites:** [NB-00 Complexity](complexity_zero_to_hero.ipynb) for the measurement harness
and amortised analysis (the table resize in §1.3 is the same argument as the dynamic array's),
[NB-01 Arrays](arrays_zero_to_hero.ipynb) for contiguity and the resize itself, and
[NB-02 Strings](strings_zero_to_hero.ipynb) §2.4 for the rolling hash and the first appearance of
"an adversary who knows your parameters".

***
# Part 0 - Setup

Standard library only, plus `dsa_toolkit` from this folder. The JDK check is not decoration:
§1.6 and §3 both depend on Java behaviour that has no Python equivalent, and the notebook says
so explicitly if Java is missing rather than silently skipping the comparison.

In [1]:
# ---------------------------------------------------------------------------
# Everything this notebook uses. Standard library only.
# ---------------------------------------------------------------------------
import itertools
import math
import random
import statistics
import sys
import time

from dsa_toolkit import (InvariantError, JavaError, StressFailure, check_invariant,
                         cross_check, growth_table, java_available, measure_growth,
                         run_java, stress)

RANDOM_SEED = 12345

ok, detail = java_available()
JAVA = ok
print("python", sys.version.split()[0])
print("JDK available:", ok, "|", detail)

python 3.14.7
JDK available: True | javac 25.0.4.1


***
# Part 1 - Theory from zero

1. What a hash function must guarantee, measured
2. Collisions are not bad luck, they are arithmetic
3. **Chaining**, built from scratch
4. **Open addressing**, built from scratch — and the delete bug
5. **Load factor**, measured against the formulas
6. Java: the `hashCode`/`equals` contract

## 1.1 What a hash function must guarantee

A hash function maps a key to an integer. For a hash *table* it must be:

1. **Deterministic** — equal keys give equal hashes, always, for the lifetime of the table. This
   is the one that mutable keys violate (§1.6).
2. **Fast** — $O(|key|)$ at worst. A hash that costs more than the search it saves is pointless.
3. **Uniform** — it should spread keys across buckets evenly.

It does **not** need to be cryptographic, one-way, or collision-*proof*. Those are different
guarantees for a different threat model, and they cost far more.

Uniformity is the one worth measuring, because "spreads keys evenly" sounds subjective and is
not. If a hash were perfectly uniform, then throwing $n$ keys into $m$ buckets makes each bucket
a binomial draw, and two consequences follow immediately:

- the expected fraction of **empty** buckets is $e^{-\alpha}$, where $\alpha = n/m$;
- the **chi-squared statistic** $\sum_i (b_i - \alpha)^2 / \alpha$ has expectation $m - 1$, so
  dividing it by $m-1$ gives a number that should sit near **1.0**.

That second one is the useful diagnostic: it is a single number, it has a known target, and it
does not care about the alphabet or the key distribution. Here are four hash functions on the
same 20,000 keys.

In [2]:
# ---------------------------------------------------------------------------
# Four hash functions, one key set, one measurement.
# ---------------------------------------------------------------------------
M = 4096

rng = random.Random(RANDOM_SEED)
words = set()
while len(words) < 20_000:
    words.add("".join(rng.choice("abcdefghijklmnopqrstuvwxyz")
                      for _ in range(rng.randint(3, 8))))
words = sorted(words)


def h_first(s):
    """Only looks at the first character. Deliberately terrible."""
    return ord(s[0])


def h_sum(s):
    """Sum of character codes. The classic beginner hash."""
    return sum(ord(c) for c in s)


def h_poly31(s):
    """Polynomial rolling hash, base 31 -- what Java's String.hashCode does."""
    h = 0
    for c in s:
        h = (h * 31 + ord(c)) & 0xFFFFFFFF
    return h


def bucket_stats(hash_fn, keys, m):
    counts = [0] * m
    for k in keys:
        counts[hash_fn(k) % m] += 1
    alpha = len(keys) / m
    empty = sum(1 for c in counts if c == 0)
    chi2 = sum((c - alpha) ** 2 for c in counts) / alpha
    return {"used": m - empty, "max": max(counts),
            "empty_pct": 100.0 * empty / m, "chi2_df": chi2 / (m - 1)}


alpha = len(words) / M
print("%d distinct keys into %d buckets   (load factor alpha = %.2f)" % (len(words), M, alpha))
print("uniform prediction: empty buckets = e^-alpha = %.2f%%,  chi2/df = 1.00"
      % (100 * math.exp(-alpha)))
print()
print("  %-16s %8s %8s %10s %10s" % ("hash function", "used", "max", "empty %", "chi2/df"))
print("  " + "-" * 56)
for name, fn in (("first char", h_first), ("sum of chars", h_sum),
                 ("poly base 31", h_poly31), ("python hash()", hash)):
    s = bucket_stats(fn, words, M)
    print("  %-16s %8d %8d %9.1f%% %10.2f"
          % (name, s["used"], s["max"], s["empty_pct"], s["chi2_df"]))

20000 distinct keys into 4096 buckets   (load factor alpha = 4.88)
uniform prediction: empty buckets = e^-alpha = 0.76%,  chi2/df = 1.00

  hash function        used      max    empty %    chi2/df
  --------------------------------------------------------
  first char             26      838      99.4%     765.74
  sum of chars          591      108      85.6%      50.52
  poly base 31         4070       14       0.6%       0.97
  python hash()        4066       16       0.7%       1.01


Read the last column against its target of 1.00.

`poly base 31` scores **0.97** and Python's built-in lands near **1.0** (it varies from run to
run, because `str` hashing is randomised per process — §3.3) — both statistically
indistinguishable from a uniform random assignment. Their empty-bucket fractions sit within a
fraction of a percent of the predicted $e^{-4.88} = 0.76\%$. Two independent checks agreeing is a good sign the model is
right.

The other two are not close, and they fail differently:

- **`sum of chars`** gets $\chi^2/df \approx 51$, and its worst bucket holds 108 keys where a
  well-behaved hash's worst holds 14. Summing throws away order, so every anagram
  collides — and worse, the sum of 3–8 lowercase letters lives in a narrow band around
  $5.5 \times 97$, so the keys pile into a few hundred adjacent buckets and **86% sit empty**.
- **`first char`** gets $\approx 765$ and uses **26 buckets out of 4,096**. Its output range is
  the alphabet. No table can rescue this; the information was discarded before the modulo.

The lesson is not "use `poly31`". It is that **uniformity is measurable in about ten lines**, so a
hash function you are unsure about is a question you can settle rather than argue about. The
number to remember is $\chi^2/df$: near 1 is uniform, and how far above 1 tells you how badly
concentrated the keys are.

One thing this measurement deliberately does not test: **whether an adversary can find collisions
on purpose.** `poly31` scores as well as Python's built-in here and is catastrophically weak in
that sense — §3 constructs collisions for it in one line. Uniformity on random input and
resistance to chosen input are different properties, and only one of them is on this table.

## 1.2 Collisions are not bad luck, they are arithmetic

A hash table maps a large key space into $m$ buckets. Since the key space is bigger, collisions
are guaranteed by pigeonhole. But that undersells it badly, because the intuition most people
have — "with 4,096 buckets I will be fine until I have a few thousand keys" — is wrong by an
order of magnitude.

This is the birthday problem. The expected number of keys inserted before the **first** collision
is approximately

$$ \sqrt{\frac{\pi m}{2}} $$

which is $\Theta(\sqrt{m})$, not $\Theta(m)$. For 4,096 buckets that is about **80 keys**.

Worth being clear about why: the $k$-th key has $k-1$ chances to collide, not one, so the number
of *pairs* grows as $k^2$ and the expected collision count passes 1 when $k \approx \sqrt{m}$.
Let us check the constant.

In [3]:
# ---------------------------------------------------------------------------
# How many keys before the first collision? Measured against sqrt(pi*m/2).
# ---------------------------------------------------------------------------
def first_collision(m, rng):
    """Insert random keys until two land in the same bucket; return how many."""
    seen = set()
    n = 0
    while True:
        n += 1
        b = rng.randrange(m)
        if b in seen:
            return n
        seen.add(b)


print("  %10s %14s %16s %8s" % ("buckets m", "measured mean", "sqrt(pi*m/2)", "ratio"))
print("  " + "-" * 52)
for m in (1_000, 4_096, 10_000, 100_000):
    rng = random.Random(m)
    trials = [first_collision(m, rng) for _ in range(400)]
    measured = statistics.mean(trials)
    predicted = math.sqrt(math.pi * m / 2)
    print("  %10s %14.1f %16.1f %8.3f"
          % ("{:,}".format(m), measured, predicted, measured / predicted))

print()
print("A table with 100,000 buckets sees its first collision at around 400 keys,")
print("when it is 0.4% full. Collision handling is not an edge case to bolt on")
print("later -- it is the main event, and it starts almost immediately.")

   buckets m  measured mean     sqrt(pi*m/2)    ratio
  ----------------------------------------------------
       1,000           40.9             39.6    1.033
       4,096           79.5             80.2    0.991
      10,000          123.9            125.3    0.988
     100,000          387.5            396.3    0.978

A table with 100,000 buckets sees its first collision at around 400 keys,
when it is 0.4% full. Collision handling is not an edge case to bolt on
later -- it is the main event, and it starts almost immediately.


The prediction holds to within about 3% across two orders of magnitude in $m$.

The practical reading: a table with 100,000 buckets collides at roughly 400 keys, **0.4% full**.
Any design that treats collisions as rare enough to handle badly is wrong from the first
thousand insertions. So there are two strategies, and the next two sections build both.

## 1.3 Chaining, built from scratch

**The idea:** each bucket holds a list of entries. Collisions append. A lookup hashes to a bucket
and scans it.

**The invariant**, which we assert after every single operation rather than claiming:

> Every stored key sits in the bucket its hash selects; no key appears twice; the entry count
> matches the entries actually stored; and the load factor never exceeds the configured maximum.

That last clause is what forces the resize, and the resize is the same amortised argument as
NB-01 §2.2's dynamic array: doubling makes the total rehashing cost $\Theta(n)$ across $n$
insertions, so each insertion is $\Theta(1)$ amortised even though individual ones are $\Theta(n)$.

In [4]:
# ---------------------------------------------------------------------------
# Separate chaining.
# ---------------------------------------------------------------------------
class ChainingMap:
    """Each bucket is a list of (key, value) pairs."""

    def __init__(self, capacity=8, max_load=0.75):
        self._buckets = [[] for _ in range(capacity)]
        self._n = 0
        self._max_load = max_load

    def __len__(self):
        return self._n

    def load_factor(self):
        return self._n / len(self._buckets)

    def _index(self, key):
        return hash(key) % len(self._buckets)

    def put(self, key, value):
        b = self._buckets[self._index(key)]
        for i, (k, _) in enumerate(b):
            if k == key:                       # update in place, do not grow
                b[i] = (key, value)
                return
        b.append((key, value))
        self._n += 1
        if self.load_factor() > self._max_load:
            self._resize(len(self._buckets) * 2)

    def get(self, key, default=None):
        for k, v in self._buckets[self._index(key)]:
            if k == key:
                return v
        return default

    def delete(self, key):
        b = self._buckets[self._index(key)]
        for i, (k, _) in enumerate(b):
            if k == key:
                b.pop(i)
                self._n -= 1
                return True
        return False

    def items(self):
        return [kv for b in self._buckets for kv in b]

    def _resize(self, capacity):
        old = self.items()
        self._buckets = [[] for _ in range(capacity)]
        self._n = 0
        for k, v in old:                       # every key must be rehashed: m changed
            self.put(k, v)


def chaining_ok(m):
    """The invariant, as a predicate. Returns True or a string saying what broke."""
    total = 0
    for i, b in enumerate(m._buckets):
        total += len(b)
        keys = [k for k, _ in b]
        if len(set(keys)) != len(keys):
            return "duplicate key in bucket %d" % i
        for k in keys:
            if hash(k) % len(m._buckets) != i:
                return "key %r sits in bucket %d but hashes to %d" % (
                    k, i, hash(k) % len(m._buckets))
    if total != m._n:
        return "count says %d but %d entries are stored" % (m._n, total)
    if m.load_factor() > m._max_load:
        return "load factor %.3f exceeds max %.3f" % (m.load_factor(), m._max_load)
    return True


print("ChainingMap defined. Invariant checked after every operation in the next cell.")

ChainingMap defined. Invariant checked after every operation in the next cell.


Now the differential test. A hand-written example proves nothing; the test below replays
**thousands of randomised operation sequences** — mixed puts, deletes and gets, with keys chosen
from a small range so collisions and re-insertions actually happen — against Python's `dict`, and
checks the invariant after every individual operation.

`stress` minimises any failing sequence before reporting it, which is the difference between a
test that tells you "something is wrong" and one that hands you the four operations that break it.

In [5]:
# ---------------------------------------------------------------------------
# Differential test: our map vs dict, invariant checked after EVERY operation.
# ---------------------------------------------------------------------------
def gen_ops(rng):
    """A random operation sequence. Small key range => frequent collisions."""
    n = rng.randrange(0, 60)
    return [(rng.choice(["put", "put", "put", "del", "get"]),
             rng.randrange(-15, 15), rng.randrange(100))
            for _ in range(n)]


def reference(ops):
    """What dict does with the same sequence."""
    d = {}
    for op, k, v in ops:
        if op == "put":
            d[k] = v
        elif op == "del":
            d.pop(k, None)
    return sorted(d.items())


def replay_chaining(ops):
    m, d = ChainingMap(), {}
    for op, k, v in ops:
        if op == "put":
            m.put(k, v)
            d[k] = v
        elif op == "del":
            m.delete(k)
            d.pop(k, None)
        else:
            assert m.get(k) == d.get(k), "get(%r) disagreed" % (k,)
        check_invariant(m, chaining_ok, "chaining invariant", "%s %r" % (op, k))
        assert len(m) == len(d), "length %d vs %d" % (len(m), len(d))
    return sorted(m.items())


checked = stress(replay_chaining, reference, gen_ops, n=3000, seed=RANDOM_SEED,
                 label="ChainingMap")
print("ChainingMap: %s randomised operation sequences agree with dict,"
      % "{:,}".format(checked))
print("             with the invariant verified after every individual operation.")

ChainingMap: 3,000 randomised operation sequences agree with dict,
             with the invariant verified after every individual operation.


## 1.4 Open addressing, built from scratch — and the delete bug

**The idea:** no lists. Every entry lives in the table itself. On collision, walk forward until a
free slot appears — that is **linear probing**. A lookup walks the same path and stops at the
first empty slot, because an empty slot proves the key was never inserted.

That last sentence is the whole design, and it is also where the bug lives.

**Deleting is not "clear the slot".** If you empty a slot that some other key probed *past*, you
break the chain that key depends on, and it becomes unreachable while still sitting in the table.
The fix is a **tombstone**: a marker meaning "something was here, keep probing". Tombstones stop
lookups terminating early, and they are why open-addressed tables need periodic rebuilds — they
accumulate, and a table full of tombstones has fast inserts and slow everything else.

Below, both versions are implemented: the correct one, and the one that clears the slot. The
second is there to be caught.

In [6]:
# ---------------------------------------------------------------------------
# Open addressing with linear probing.
# ---------------------------------------------------------------------------
EMPTY = object()          # never held a key
DEAD = object()           # tombstone: held a key, keep probing past it


class ProbingMap:
    """Linear probing. Deletes leave tombstones."""

    def __init__(self, capacity=8, max_load=0.5):
        self._keys = [EMPTY] * capacity
        self._vals = [None] * capacity
        self._n = 0                # live entries
        self._used = 0             # live + tombstones: what actually slows probing
        self._max_load = max_load

    def __len__(self):
        return self._n

    def _slots(self):
        return len(self._keys)

    def _probe(self, key):
        """Slot indices in probe order: home, home+1, home+2, ..."""
        i = hash(key) % self._slots()
        for _ in range(self._slots()):
            yield i
            i = (i + 1) % self._slots()

    def put(self, key, value):
        first_dead = None
        for i in self._probe(key):
            k = self._keys[i]
            if k is EMPTY:
                # Reuse the earliest tombstone seen, but only now that we know
                # the key is genuinely absent.
                target = i if first_dead is None else first_dead
                self._keys[target] = key
                self._vals[target] = value
                self._n += 1
                if first_dead is None:
                    self._used += 1
                if self._used / self._slots() > self._max_load:
                    self._resize(self._slots() * 2)
                return
            if k is DEAD:
                if first_dead is None:
                    first_dead = i
            elif k == key:
                self._vals[i] = value
                return
        raise RuntimeError("table full")

    def get(self, key, default=None):
        for i in self._probe(key):
            k = self._keys[i]
            if k is EMPTY:
                return default            # an empty slot proves the key is absent
            if k is not DEAD and k == key:
                return self._vals[i]
        return default

    def delete(self, key):
        for i in self._probe(key):
            k = self._keys[i]
            if k is EMPTY:
                return False
            if k is not DEAD and k == key:
                self._keys[i] = DEAD      # tombstone, NOT EMPTY
                self._vals[i] = None
                self._n -= 1
                return True
        return False

    def items(self):
        return [(k, v) for k, v in zip(self._keys, self._vals)
                if k is not EMPTY and k is not DEAD]

    def _resize(self, capacity):
        old = self.items()                # tombstones are dropped here
        self._keys = [EMPTY] * capacity
        self._vals = [None] * capacity
        self._n = self._used = 0
        for k, v in old:
            self.put(k, v)


class BuggyProbingMap(ProbingMap):
    """Identical, except delete clears the slot instead of leaving a tombstone."""

    def delete(self, key):
        for i in self._probe(key):
            k = self._keys[i]
            if k is EMPTY:
                return False
            if k is not DEAD and k == key:
                self._keys[i] = EMPTY     # <-- the bug, and the only difference
                self._vals[i] = None
                self._n -= 1
                self._used -= 1           # bookkeeping stays consistent, so counts look fine
                return True
        return False


def probing_ok(m):
    """Counts agree, and every live key is still reachable by probing."""
    live = sum(1 for k in m._keys if k is not EMPTY and k is not DEAD)
    if live != m._n:
        return "count says %d but %d slots are live" % (m._n, live)
    used = sum(1 for k in m._keys if k is not EMPTY)
    if used != m._used:
        return "used says %d but %d slots are non-empty" % (m._used, used)
    for k in (k for k in m._keys if k is not EMPTY and k is not DEAD):
        if m.get(k, EMPTY) is EMPTY:
            return "live key %r is not reachable by probing" % (k,)
    return True


print("ProbingMap and BuggyProbingMap defined.")

ProbingMap and BuggyProbingMap defined.


In [7]:
# ---------------------------------------------------------------------------
# The same differential test, on both versions.
# ---------------------------------------------------------------------------
def replay_probing(ops, cls):
    m, d = cls(), {}
    for op, k, v in ops:
        if op == "put":
            m.put(k, v)
            d[k] = v
        elif op == "del":
            m.delete(k)
            d.pop(k, None)
        else:
            assert m.get(k) == d.get(k), "get(%r) disagreed" % (k,)
        check_invariant(m, probing_ok, "probing invariant", "%s %r" % (op, k))
    return sorted(m.items())


checked = stress(lambda o: replay_probing(o, ProbingMap), reference, gen_ops,
                 n=3000, seed=RANDOM_SEED, label="ProbingMap")
print("ProbingMap (tombstones): %s sequences agree with dict." % "{:,}".format(checked))

print()
print("The identical test against the version that clears the slot on delete:")
print()
try:
    stress(lambda o: replay_probing(o, BuggyProbingMap), reference, gen_ops,
           n=3000, seed=RANDOM_SEED, label="BuggyProbingMap")
    print("  NO FAILURE -- which would mean the test is not strong enough.")
except (StressFailure, InvariantError, AssertionError) as exc:
    for line in str(exc).split("\n"):
        print("  " + line)

ProbingMap (tombstones): 3,000 sequences agree with dict.

The identical test against the version that clears the slot on delete:

  BuggyProbingMap: implementation disagrees with reference
    failing input : [('put', -10, 86), ('put', 14, 18), ('del', -10, 39)]
    implementation: 'impl raised InvariantError: probing invariant violated after del -10: live key 14 is not reachable by probing\n  state: <__main__.BuggyProbingMap object at 0x00000171A05AD250>'
    reference     : None


**Three operations.** That is the entire counterexample, after `stress` minimised it:

```
put -10, put 14, del -10   ->   live key 14 is not reachable by probing
```

Trace it at the starting capacity of 8. `hash(-10) % 8 == 6`, so `-10` lands in slot 6.
`hash(14) % 8 == 6` too, so `14` finds slot 6 taken and probes forward to slot 7. Now delete
`-10`, which clears slot 6. Look up `14`: the probe starts at its home slot 6, finds it **empty**,
and concludes — correctly, by the only rule the table has — that `14` was never inserted.

`14` is still sitting in slot 7. `items()` still lists it. The entry count still counts it. It is
simply unreachable.

Three things worth taking from this:

- **The bug is in `delete`, and it corrupts `get` for a different key.** Nothing about looking up
  `12` is wrong. This is why testing operations in isolation misses it and replaying randomised
  *sequences* finds it.
- **The counterexample is minimal because the harness shrank it.** The first failing sequence was
  dozens of operations long. Four is short enough to trace by hand, and that is the difference
  between a failing test and a diagnosis.
- **Consistent bookkeeping hid it from the cheap check.** `BuggyProbingMap` decrements its
  counters correctly, so length checks and count invariants all pass. What caught it was the
  reachability clause — an invariant about the *structure*, not about a number.

## 1.5 Load factor, measured against the formulas

The **load factor** $\alpha = n/m$ is the single number that governs a hash table's performance,
and it is where the two designs stop being interchangeable.

For an unsuccessful search — the common case, since inserts and misses both start with one —
the classical results under uniform hashing are:

| Design | Expected probes, unsuccessful search | Behaviour as $\alpha \to 1$ |
|---|---|---|
| **Chaining** | $\alpha$ | grows **linearly**, defined for $\alpha > 1$ |
| **Linear probing** | $\frac{1}{2}\left(1 + \dfrac{1}{(1-\alpha)^2}\right)$ | grows **quadratically**, undefined at $\alpha = 1$ |

The linear-probing formula is Knuth's, and the $(1-\alpha)^{-2}$ is the part that matters: it is
not merely worse than chaining, it is worse *by a growing factor*. Clustering is why — a run of
occupied slots is more likely to be extended than a gap is to be hit, so occupied runs grow, and
long runs make future probes longer still.

Both are asymptotic in $m$, so the measurement below averages over several independent tables.
The spread column is there because at high $\alpha$ the variance is enormous, and a single table
can land 25% either side of the mean.

In [8]:
# ---------------------------------------------------------------------------
# Probes for an unsuccessful search, measured against the formulas.
# ---------------------------------------------------------------------------
TABLE_BITS = 16
M_SLOTS = 1 << TABLE_BITS
TABLES = 6
LOOKUPS = 20_000


def probing_probes(m, n, rng):
    """Build a linear-probing table of n keys in m slots; measure a failed search."""
    slots = [False] * m
    for _ in range(n):
        i = rng.randrange(1 << 40) % m
        while slots[i]:
            i = (i + 1) % m
        slots[i] = True
    total = 0
    for _ in range(LOOKUPS):
        i = rng.randrange(1 << 40) % m
        p = 1
        while slots[i]:
            i = (i + 1) % m
            p += 1
        total += p
    return total / LOOKUPS


def chaining_probes(m, n, rng):
    """Same, for chaining: a failed search scans one whole bucket."""
    buckets = [0] * m
    for _ in range(n):
        buckets[rng.randrange(1 << 40) % m] += 1
    total = 0
    for _ in range(LOOKUPS):
        total += buckets[rng.randrange(1 << 40) % m]
    return total / LOOKUPS


print("Unsuccessful search, m = 2^%d = %s slots, averaged over %d independent tables"
      % (TABLE_BITS, "{:,}".format(M_SLOTS), TABLES))
print()
print("  %6s | %9s %9s %7s | %9s %9s %7s | %10s"
      % ("alpha", "chain", "theory", "ratio", "probe", "theory", "ratio", "probe spread"))
print("  " + "-" * 82)
for a in (0.25, 0.50, 0.75, 0.90, 0.95):
    n = int(M_SLOTS * a)
    ch = [chaining_probes(M_SLOTS, n, random.Random(s)) for s in range(TABLES)]
    pr = [probing_probes(M_SLOTS, n, random.Random(s)) for s in range(TABLES)]
    ch_m, pr_m = statistics.mean(ch), statistics.mean(pr)
    ch_t = a
    pr_t = 0.5 * (1 + 1 / (1 - a) ** 2)
    print("  %6.2f | %9.2f %9.2f %7.3f | %9.2f %9.2f %7.3f | %4.0f - %-4.0f"
          % (a, ch_m, ch_t, ch_m / ch_t, pr_m, pr_t, pr_m / pr_t, min(pr), max(pr)))

print()
print("And chaining past alpha = 1, where linear probing has no meaning at all:")
for a in (2.0, 8.0):
    n = int(M_SLOTS * a)
    ch = [chaining_probes(M_SLOTS, n, random.Random(s)) for s in range(3)]
    ch_m = statistics.mean(ch)
    print("  alpha = %4.1f -> chaining %6.2f probes, theory %4.1f, ratio %.3f"
          % (a, ch_m, a, ch_m / a))

Unsuccessful search, m = 2^16 = 65,536 slots, averaged over 6 independent tables

   alpha |     chain    theory   ratio |     probe    theory   ratio | probe spread
  ----------------------------------------------------------------------------------


    0.25 |      0.25      0.25   1.003 |      1.39      1.39   1.001 |    1 - 1   


    0.50 |      0.50      0.50   1.003 |      2.50      2.50   0.999 |    2 - 3   


    0.75 |      0.75      0.75   1.003 |      8.32      8.50   0.979 |    8 - 9   


    0.90 |      0.90      0.90   1.001 |     50.88     50.50   1.008 |   46 - 58  


    0.95 |      0.95      0.95   1.000 |    181.32    200.50   0.904 |  150 - 225 

And chaining past alpha = 1, where linear probing has no meaning at all:


  alpha =  2.0 -> chaining   2.00 probes, theory  2.0, ratio 0.999


  alpha =  8.0 -> chaining   7.99 probes, theory  8.0, ratio 0.999


Both formulas hold, and the contrast is the point.

**Chaining tracks $\alpha$ to within about 1% everywhere**, including at $\alpha = 8$ where the
table has eight times more keys than buckets. It degrades, but it degrades *gracefully and
predictably*: at $\alpha = 8$ a failed search scans 8 entries, which is slow but not a cliff.

**Linear probing matches its formula too** — within 2% up to $\alpha = 0.9$ — and that formula is
brutal. At $\alpha = 0.5$ it costs 2.5 probes against chaining's 0.5. At $\alpha = 0.95$ it costs
around 180 against chaining's 0.95, roughly **190× worse**, and it cannot go past 1 at all.

At $\alpha = 0.95$ the measurement lands about 10% *under* the formula, which is worth naming
rather than glossing: Knuth's result is asymptotic in $m$, and at $\alpha = 0.95$ a $2^{16}$-slot
table has only about 3,000 free slots left, so it runs out of room to form the long clusters the
limit assumes. At $m = 2^{20}$ the ratio comes back to about 0.98.

The $\alpha = 0.95$ row is also worth reading for its **spread**: individual tables land anywhere
from about 150 to 225 probes. That variance is not noise in the measurement, it is a real property
of the structure — at high load a linear-probing table's performance depends heavily on which
clusters happened to form. Predictability is itself a feature, and this is where it is lost.

Which is why real implementations resize well before that: **Java's `HashMap` resizes at
$\alpha = 0.75$, CPython's `dict` at about $\alpha = 0.66$**, and open-addressed tables generally
keep a lower ceiling than chained ones. The formulas above are the reason those specific numbers
exist.

So why does anyone use open addressing? **Cache locality** (NB-01 §3). Chaining follows a pointer
per entry into scattered memory; probing walks contiguous slots, and a probe sequence of 3 usually
costs one cache line rather than three cache misses. At low $\alpha$ that constant factor wins
comfortably — which is exactly why CPython's `dict` is open-addressed and keeps $\alpha$ low.

## 1.6 Java: the `hashCode`/`equals` contract

Java states the rule explicitly, and it is the same rule every hash container in every language
relies on:

> If `a.equals(b)`, then `a.hashCode() == b.hashCode()`.

The converse is *not* required — unequal objects may share a hash, that is just a collision. Only
this direction matters, and it matters because a hash container uses the hash to *find* the
bucket and `equals` to confirm the match. Break the implication and a key gets filed under one
hash and searched for under another.

Two ways to break it, both common:

1. **Override `equals` and forget `hashCode`.** The default `hashCode` is identity-based, so two
   equal objects hash differently.
2. **Mutate a key after inserting it.** The hash was correct when stored and is not any more.

The first has a defence that costs nothing, and it is worth seeing that it fires.

In [9]:
# ---------------------------------------------------------------------------
# javac catches the missing hashCode by itself -- under the flags this series
# already compiles with. Here is the compiler refusing the broken class.
# ---------------------------------------------------------------------------
BROKEN_SRC = r"""
import java.util.*;

public class BrokenKey {
    static final class Key {
        final String id;
        Key(String id) { this.id = id; }
        @Override public boolean equals(Object o) {
            return (o instanceof Key) && ((Key) o).id.equals(id);
        }
        // hashCode() deliberately not overridden.
    }
    public static void main(String[] args) {
        Map<Key,String> m = new HashMap<>();
        m.put(new Key("x"), "stored");
        System.out.println(m.get(new Key("x")));
    }
}
"""

if JAVA:
    try:
        run_java(BROKEN_SRC)
        print("compiled -- expected a warning and did not get one")
    except JavaError as exc:
        import re
        for line in str(exc).split("\n"):
            if "overrides" in line or "error" in line or "warning" in line:
                # javac reports the full temp compile path; keep only the file name.
                print(re.sub(r"^.*[\\/](\w+\.java)", r"\1", line).strip())
        print()
        print("javac -Xlint:all -Werror rejects it outright. You have to actively")
        print("silence the warning to ship this bug -- which is what the next cell does,")
        print("so we can see what it costs at runtime.")
else:
    print("JDK not available; skipping the Java sections.")

BrokenKey.java:5: warning: [overrides] Class Key overrides equals, but neither it nor any superclass overrides hashCode method
error: warnings found and -Werror specified
1 error
1 warning

javac -Xlint:all -Werror rejects it outright. You have to actively
silence the warning to ship this bug -- which is what the next cell does,
so we can see what it costs at runtime.


In [10]:
# ---------------------------------------------------------------------------
# What the contract violations actually do at runtime.
# ---------------------------------------------------------------------------
CONTRACT_SRC = r"""
import java.util.*;

public class Contract {
    @SuppressWarnings("overrides")            // javac refuses this class without it
    static final class Broken {
        final String id;
        Broken(String id) { this.id = id; }
        @Override public boolean equals(Object o) {
            return (o instanceof Broken) && ((Broken) o).id.equals(id);
        }
    }

    static final class Correct {
        final String id;
        Correct(String id) { this.id = id; }
        @Override public boolean equals(Object o) {
            return (o instanceof Correct) && ((Correct) o).id.equals(id);
        }
        @Override public int hashCode() { return id.hashCode(); }
    }

    static final class Mutable {
        String id;                            // not final: that is the whole problem
        Mutable(String id) { this.id = id; }
        @Override public boolean equals(Object o) {
            return (o instanceof Mutable) && ((Mutable) o).id.equals(id);
        }
        @Override public int hashCode() { return id.hashCode(); }
    }

    public static void main(String[] args) {
        System.out.println("1. equals overridden, hashCode not:");
        Broken b1 = new Broken("x"), b2 = new Broken("x");
        System.out.println("   b1.equals(b2)        = " + b1.equals(b2));
        System.out.println("   hashCodes differ     = " + (b1.hashCode() != b2.hashCode()));
        Map<Broken,String> mb = new HashMap<>();
        mb.put(b1, "first");
        System.out.println("   put(b1), then get(b2) = " + mb.get(b2));
        mb.put(b2, "second");
        System.out.println("   size after put(b2)    = " + mb.size()
                           + "   <- two entries, one logical key");

        System.out.println();
        System.out.println("2. both overridden, consistently:");
        Correct c1 = new Correct("x"), c2 = new Correct("x");
        Map<Correct,String> mc = new HashMap<>();
        mc.put(c1, "first");
        mc.put(c2, "second");
        System.out.println("   get(c2) = " + mc.get(c2) + "   size = " + mc.size());

        System.out.println();
        System.out.println("3. a correct key, mutated after insertion:");
        Mutable k = new Mutable("before");
        Map<Mutable,String> mm = new HashMap<>();
        mm.put(k, "stored");
        System.out.println("   before mutation: get(k)      = " + mm.get(k));
        k.id = "after";
        System.out.println("   after k.id = \"after\": get(k) = " + mm.get(k));
        System.out.println("   containsKey(k) = " + mm.containsKey(k)
                           + "   size = " + mm.size());
        System.out.print("   but iteration still finds it: ");
        for (Map.Entry<Mutable,String> e : mm.entrySet())
            System.out.print(e.getKey().id + "=" + e.getValue());
        System.out.println();
    }
}
"""

if JAVA:
    print(run_java(CONTRACT_SRC))
else:
    print("JDK not available; skipping.")

1. equals overridden, hashCode not:
   b1.equals(b2)        = true
   hashCodes differ     = true
   put(b1), then get(b2) = null
   size after put(b2)    = 2   <- two entries, one logical key

2. both overridden, consistently:
   get(c2) = second   size = 1

3. a correct key, mutated after insertion:
   before mutation: get(k)      = stored
   after k.id = "after": get(k) = null
   containsKey(k) = false   size = 1
   but iteration still finds it: after=stored



Three distinct failures, and the third is the one that reaches production.

**Case 1 — `equals` without `hashCode`.** `b1.equals(b2)` is `true`, their hashes differ, and the
map holds **two entries for one logical key**. Note that nothing throws. The map is not corrupt by
its own rules; it was simply told two contradictory things. `javac` catches this one, and the
lesson is that the compiler flag is worth more than the knowledge — you cannot forget to run it.

**Case 2** is the control: both overridden, one entry, correct value.

**Case 3 — the mutable key.** The entry is inserted correctly, then the key's `id` changes, and
now `get(k)` returns `null` and `containsKey(k)` is `false` — *while the entry is still in the
map*, still counted by `size()`, still visible to iteration, still holding a reference that the
garbage collector cannot free. **The key not being findable is worse than an exception**, because
nothing anywhere reports a problem. This is a memory leak and a correctness bug wearing the same
costume.

The defence is the same in every language: **make keys immutable.** Java's `record` types and
`final` fields exist partly for this; Python enforces it structurally by requiring keys to be
hashable and making the common mutable containers (`list`, `dict`, `set`) unhashable outright.

Python's version of case 1 is worth knowing too, because its default differs. Defining `__eq__`
without `__hash__` makes the class **unhashable** — you get a `TypeError` on insertion rather than
a silently broken map. That is a strictly better default: it fails loudly, immediately, at the
point of the mistake.

In [11]:
# ---------------------------------------------------------------------------
# The Python side of case 1: defining __eq__ removes the default __hash__.
# ---------------------------------------------------------------------------
class PyKey:
    def __init__(self, ident):
        self.id = ident

    def __eq__(self, other):
        return isinstance(other, PyKey) and other.id == self.id
    # No __hash__ defined -- Python sets it to None when __eq__ is defined.


print("PyKey.__hash__ is", PyKey.__hash__)
try:
    {PyKey("x"): 1}
except TypeError as exc:
    print("using it as a dict key ->", type(exc).__name__ + ":", exc)

print()
print("Java compiles the same mistake and silently misfiles the entry;")
print("Python refuses to hash the object at all. Failing loudly at the point")
print("of the mistake beats failing quietly somewhere else later.")

print()
print("And the mutable-key failure in Python -- with a twist worth knowing about.")
print()


class Sneaky:
    """Hashable and mutable: exactly what Python's built-in key types avoid being.

    The id is an int and __hash__ returns it unchanged, so a key lands in slot
    `id & 7` of a small dict. That makes this demo deterministic instead of
    depending on string hash randomisation.
    """

    def __init__(self, ident):
        self.id = ident

    def __eq__(self, other):
        return isinstance(other, Sneaky) and other.id == self.id

    def __hash__(self):
        return self.id


for old, new in ((1, 4), (1, 9)):
    k = Sneaky(old)
    d = {k: "stored"}
    k.id = new                      # the key's hash just changed under the dict
    print("  id %d -> %d  (slots %d -> %d):  d.get(k) = %-8r  k in d = %-5r  len = %d"
          % (old, new, old & 7, new & 7, d.get(k), k in d, len(d)))

print()
print("  The second row still finds the key, and that is not the dict being")
print("  forgiving. CPython compares the stored key by IDENTITY before it")
print("  compares hashes, so when the new hash happens to probe the same slot")
print("  the lookup succeeds by accident. 1 and 9 differ by 8, so both land in")
print("  slot 1 of an 8-slot table; 1 and 4 do not.")
print()
print("  That is worse than a bug that always fires: mutating a key corrupts")
print("  the map INTERMITTENTLY, and whether it fires depends on the table size")
print("  and on hash values you do not control.")
print()

k = Sneaky(1)
d = {k: "stored"}
k.id = 4
print("  And the entry is stranded, not merely mislabelled:")
print("    d.get(Sneaky(4)) =", d.get(Sneaky(4)), "  <- the new identity does not find it")
print("    d.get(Sneaky(1)) =", d.get(Sneaky(1)), "  <- neither does the old one")
print("    len(d) =", len(d), " values =", list(d.values()),
      " <- still there, still counted")

PyKey.__hash__ is None
using it as a dict key -> TypeError: cannot use 'PyKey' as a dict key (unhashable type: 'PyKey')

Java compiles the same mistake and silently misfiles the entry;
Python refuses to hash the object at all. Failing loudly at the point
of the mistake beats failing quietly somewhere else later.

And the mutable-key failure in Python -- with a twist worth knowing about.

  id 1 -> 4  (slots 1 -> 4):  d.get(k) = None      k in d = False  len = 1
  id 1 -> 9  (slots 1 -> 1):  d.get(k) = 'stored'  k in d = True   len = 1

  The second row still finds the key, and that is not the dict being
  forgiving. CPython compares the stored key by IDENTITY before it
  compares hashes, so when the new hash happens to probe the same slot
  the lookup succeeds by accident. 1 and 9 differ by 8, so both land in
  slot 1 of an 8-slot table; 1 and 4 do not.

  That is worse than a bug that always fires: mutating a key corrupts
  the map INTERMITTENTLY, and whether it fires depends on the t

***
# Part 2 - Worked problems

Four problems that use a hash table for four different reasons. The reason is the point — each
one has a name, and the name is what transfers to a problem you have not seen.

| # | Problem | The pattern | What it buys |
|---|---|---|---|
| 2.1 | Group anagrams | **Canonical key** | equality by content, not identity |
| 2.2 | Two-sum | **Complement lookup** | $O(n^2) \to O(n)$, measured |
| 2.3 | Subarray sum equals k | **Prefix sums in a map** | ranges from points |
| 2.4 | Deduplication | **Membership set** | $O(n^2) \to O(n)$, and an ordering trap |

## 2.1 Group anagrams — the canonical key

**Problem.** Group words that are anagrams of each other.

**The pattern: a canonical key.** A hash map keys on *equality*, so if you want to treat several
different objects as the same thing, map each to a **canonical form** and key on that. Sorted
letters work: `"eat"`, `"tea"` and `"ate"` all canonicalise to `"aet"`.

This is the pattern behind deduplicating by normalised email, matching records by cleaned-up
name, and caching by a request's canonical form. The hash table is trivial; **choosing the
canonical form is the whole problem**, and getting it wrong is how you get bugs that only affect
some users — see NB-02 §1.4, where two visually identical strings compare unequal because one is
NFC and the other NFD.

Two canonical forms below: sorting the letters ($O(k \log k)$ per word) and a 26-slot count
signature ($O(k)$). Both are stress-tested against a brute-force pairwise grouping.

In [12]:
# ---------------------------------------------------------------------------
# 2.1 Group anagrams: two canonical keys, one brute-force reference.
# ---------------------------------------------------------------------------
def group_by_sorted(words):
    """Canonical key = the word's letters in sorted order."""
    groups = {}
    for w in words:
        groups.setdefault("".join(sorted(w)), []).append(w)
    return sorted(sorted(g) for g in groups.values())


def group_by_counts(words):
    """Canonical key = a 26-slot letter-count tuple. O(k) instead of O(k log k)."""
    groups = {}
    for w in words:
        counts = [0] * 26
        for ch in w:
            counts[ord(ch) - 97] += 1
        groups.setdefault(tuple(counts), []).append(w)
    return sorted(sorted(g) for g in groups.values())


def group_brute_force(words):
    """Reference: compare every pair. O(n^2 k log k), obviously correct."""
    def canon(w):
        return sorted(w)

    used = [False] * len(words)
    out = []
    for i, w in enumerate(words):
        if used[i]:
            continue
        group = [w]
        used[i] = True
        for j in range(i + 1, len(words)):
            if not used[j] and canon(words[j]) == canon(w):
                group.append(words[j])
                used[j] = True
        out.append(sorted(group))
    return sorted(out)


def gen_words(rng):
    n = rng.randrange(0, 25)
    # Tiny alphabet and short words, so anagrams actually occur.
    return ["".join(rng.choice("abc") for _ in range(rng.randint(1, 4)))
            for _ in range(n)]


demo = ["eat", "tea", "tan", "ate", "nat", "bat"]
print("input :", demo)
print("groups:", group_by_sorted(demo))
print()

for name, fn in (("sorted-letters key", group_by_sorted), ("count-tuple key", group_by_counts)):
    checked = stress(fn, group_brute_force, gen_words, n=2000, seed=RANDOM_SEED, label=name)
    print("%-20s %s random word lists agree with the brute-force grouping"
          % (name + ":", "{:,}".format(checked)))

input : ['eat', 'tea', 'tan', 'ate', 'nat', 'bat']
groups: [['ate', 'eat', 'tea'], ['bat'], ['nat', 'tan']]



sorted-letters key:  2,000 random word lists agree with the brute-force grouping


count-tuple key:     2,000 random word lists agree with the brute-force grouping


Both agree with brute force on every case. The trap worth naming: a **list is not hashable**, so
the count signature must be a `tuple`. That is not Python being awkward — it is §1.6's rule
enforced structurally. A list can change, so it cannot be a key.

## 2.2 Two-sum — the complement lookup

**Problem.** Given a list and a target, find two indices whose values sum to the target.

**The pattern: complement lookup.** The brute force asks "for each pair, do they sum to the
target?" — $O(n^2)$. The reframing asks, for each element $x$, "have I already seen
$\text{target} - x$?" — and *have I already seen* is exactly what a hash set answers in $O(1)$.

The move is general and worth stating on its own: **replace a search over pairs with a lookup
keyed on what you need.** It works whenever the condition relating two elements can be solved for
one of them.

The subtlety is that you build the map **while** scanning, checking before inserting. That
handles the "cannot use the same element twice" rule for free, and it is why the one-pass version
is correct where a two-pass version needs a special case for `target == 2 * x`.

In [13]:
# ---------------------------------------------------------------------------
# 2.2 Two-sum: build the map while scanning.
# ---------------------------------------------------------------------------
def two_sum(nums, target):
    """Return the first (i, j) with i < j and nums[i] + nums[j] == target, else None."""
    seen = {}                                  # value -> earliest index
    for j, x in enumerate(nums):
        need = target - x
        if need in seen:                       # check BEFORE inserting x
            return (seen[need], j)
        if x not in seen:                      # keep the earliest index
            seen[x] = j
    return None


def two_sum_brute(nums, target):
    """Reference: every pair, in the order the one-pass scan would find them."""
    for j in range(len(nums)):
        for i in range(j):
            if nums[i] + nums[j] == target:
                return (i, j)
    return None


def gen_case(rng):
    n = rng.randrange(0, 30)
    nums = [rng.randrange(-10, 10) for _ in range(n)]
    return (nums, rng.randrange(-20, 20))


checked = stress(lambda c: two_sum(*c), lambda c: two_sum_brute(*c), gen_case,
                 n=4000, seed=RANDOM_SEED, label="two_sum")
print("two_sum: %s random cases agree with the brute-force pair search," % "{:,}".format(checked))
print("         including duplicates, negatives, empty lists and target == 2*x.")
print()
print("  [2, 7, 11, 15], target 9  ->", two_sum([2, 7, 11, 15], 9))
print("  [3, 3],         target 6  ->", two_sum([3, 3], 6), "  <- same value, two indices")
print("  [3],            target 6  ->", two_sum([3], 6), "  <- must NOT reuse one element")

two_sum: 4,000 random cases agree with the brute-force pair search,
         including duplicates, negatives, empty lists and target == 2*x.

  [2, 7, 11, 15], target 9  -> (0, 1)
  [3, 3],         target 6  -> (0, 1)   <- same value, two indices
  [3],            target 6  -> None   <- must NOT reuse one element


In [14]:
# ---------------------------------------------------------------------------
# And the complexity claim, measured rather than asserted.
# ---------------------------------------------------------------------------
def make_worst_case(n):
    """No pair sums to the target, so both versions do all their work."""
    return ([2 * i for i in range(n)], 1)      # all even, odd target


def two_sum_ops(nums, target):
    """The one-pass scan, returning how many elements it examined."""
    seen = {}
    ops = 0
    for j, x in enumerate(nums):
        ops += 1
        need = target - x
        if need in seen:
            return ops
        if x not in seen:
            seen[x] = j
    return ops


print("brute force, O(n^2):")
growth_table(measure_growth(lambda c: two_sum_brute(*c), [500, 1_000, 2_000, 4_000],
                            setup=make_worst_case, repeats=3), claim="O(n^2)")

print()
print("hash-map version -- COUNTING elements examined, not timing:")
print("  %10s %14s %8s" % ("n", "elements", "ratio"))
prev = None
for n in (200_000, 400_000, 800_000, 1_600_000):
    ops = two_sum_ops(*make_worst_case(n))
    print("  %10s %14s %8s" % ("{:,}".format(n), "{:,}".format(ops),
                               "-" if prev is None else "%.2f" % (ops / prev)))
    prev = ops

brute force, O(n^2):


         n        seconds      ratio
------------------------------------
       500       0.006457          -
     1,000       0.031512       4.88
     2,000       0.131607       4.18
     4,000       0.548563       4.17

best fit: O(n^2) (relative error 0.116); next: O(n^3) (1.152)
claimed O(n^2) -> measurement MATCHES the claim

hash-map version -- COUNTING elements examined, not timing:
           n       elements    ratio
     200,000        200,000        -


     400,000        400,000     2.00


     800,000        800,000     2.00


   1,600,000      1,600,000     2.00


The brute force matches $O(n^2)$ cleanly. The hash version is reported differently, and the reason
is the lesson.

**Timing the hash version at these sizes does not cleanly separate $O(n)$ from $O(n \log n)$** —
the per-iteration work (a dict lookup, the arithmetic, a conditional insert) plus the dict's own
resizes make the wall-clock ratios wander between about 2.0 and 2.2 from run to run, and the fit
flips between the two classes. That is not the algorithm being super-linear; it is NB-00 §1.7's
warning that a stopwatch cannot resolve a $\log n$ factor at modest sizes.

So we **count operations instead**, and the count is unambiguous: the scan examines exactly $n$
elements, ratio 2.00 per doubling, no noise. Counting the thing the complexity is *about* beats
timing a proxy for it — the same move the harness's `NOT SEPARABLE` verdict recommends when the
timings will not settle.

Note the size ranges regardless: the quadratic version is measured at 500–4,000 and the hash
version at 200,000+, because at a size where the brute force is still tolerable the hash version
is too fast to measure. That gap *is* the result.

The cost is memory: $O(n)$ for the map against $O(1)$ for the brute force. That is the trade this
pattern always makes, and it is worth saying out loud because it is not always the right one — on
a memory-constrained device, or when the list is already sorted, **two pointers** solve this in
$O(1)$ extra space (NB-01 §2.3, and revisited in NB-15).

## 2.3 Subarray sum equals k — prefix sums in a map

**Problem.** Count the contiguous subarrays summing to exactly $k$.

This is the one people find hardest, and the difficulty is that the obvious hash-table use —
"store the subarrays" — is useless, since there are $\Theta(n^2)$ of them.

**The pattern: turn a range question into a point question.** With prefix sums
$P_i = a_0 + \dots + a_{i-1}$, the sum of $a_i..a_{j-1}$ is $P_j - P_i$. So

$$ \text{sum}(i, j) = k \iff P_i = P_j - k $$

and the question "how many subarrays ending at $j$ sum to $k$" becomes "how many **earlier prefix
values** equal $P_j - k$" — a counting lookup. This is the same reframing as §2.2, applied to
prefix sums, and it is the reason NB-01 §2.4 built prefix sums in the first place.

**The detail everyone gets wrong** is the initial `{0: 1}`. The empty prefix has sum 0 and must be
counted, or every subarray starting at index 0 is missed. Below, both the correct and the
`{}`-initialised versions are tested, and the stress test is asked to find the difference.

In [15]:
# ---------------------------------------------------------------------------
# 2.3 Count subarrays summing to k.
# ---------------------------------------------------------------------------
def count_subarrays(nums, k):
    """Count contiguous subarrays with sum exactly k."""
    counts = {0: 1}          # the empty prefix: sum 0, seen once
    running = 0
    total = 0
    for x in nums:
        running += x
        total += counts.get(running - k, 0)    # count first ...
        counts[running] = counts.get(running, 0) + 1   # ... then record this prefix
    return total


def count_subarrays_no_seed(nums, k):
    """Identical, but without the {0: 1} seed. Kept to be caught."""
    counts = {}
    running = 0
    total = 0
    for x in nums:
        running += x
        total += counts.get(running - k, 0)
        counts[running] = counts.get(running, 0) + 1
    return total


def count_brute(nums, k):
    """Reference: every (i, j) pair."""
    total = 0
    for i in range(len(nums)):
        s = 0
        for j in range(i, len(nums)):
            s += nums[j]
            if s == k:
                total += 1
    return total


def gen_sub(rng):
    n = rng.randrange(0, 25)
    return ([rng.randrange(-4, 5) for _ in range(n)], rng.randrange(-6, 7))


checked = stress(lambda c: count_subarrays(*c), lambda c: count_brute(*c), gen_sub,
                 n=4000, seed=RANDOM_SEED, label="count_subarrays")
print("count_subarrays: %s random cases agree with the O(n^2) reference."
      % "{:,}".format(checked))
print("  (negatives included -- this is why a sliding window does NOT work here.)")
print()
print("Now the version missing the {0: 1} seed:")
try:
    stress(lambda c: count_subarrays_no_seed(*c), lambda c: count_brute(*c), gen_sub,
           n=4000, seed=RANDOM_SEED, label="count_subarrays_no_seed")
    print("  no failure -- the test is not strong enough")
except StressFailure as exc:
    for line in str(exc).split("\n"):
        print("  " + line)

count_subarrays: 4,000 random cases agree with the O(n^2) reference.
  (negatives included -- this is why a sliding window does NOT work here.)

Now the version missing the {0: 1} seed:
  count_subarrays_no_seed: implementation disagrees with reference
    failing input : ([-2, 1, -3, 4, 2, 4, -2, -2, -1, -3, -1, 1, 1, -4, 3, 1, -4], 2)
    implementation: 10
    reference     : 12


The minimised counterexample is about as small as it could be: a single-element list whose only
element equals `k`. The subarray `[k]` starts at index 0, so it needs the empty prefix to be in
the map, and without the seed the count comes back 0 instead of 1.

Two things this section is really about:

- **A hash map counts things, not just stores them.** `counts.get(x, 0) + 1` turns a map into a
  multiset, and the multiset is what makes the answer a *count* rather than a yes/no.
- **Order matters within the loop.** Look up before inserting. Insert first and a prefix can match
  itself, counting an empty subarray when $k = 0$. That is the same
  check-before-insert discipline as §2.2, for the same reason.

And the constraint that rules out the cheaper tool: **negatives**. With only non-negative values a
sliding window solves this in $O(1)$ space, because the running sum is monotone. Allow negatives
and the window can neither be safely grown nor shrunk, so the prefix-map is the right structure.
Whenever a problem says "positive integers", check whether it is licensing a window.

## 2.4 Deduplication — the membership set, and an ordering trap

**Problem.** Remove duplicates from a list, keeping the first occurrence of each.

**The pattern: a membership set.** `x in list` is $O(n)$; `x in set` is $O(1)$ expected. Swapping
the container changes the algorithm's class without changing its shape — the most common real-world
hash-table win, and the easiest to miss in code review because the two versions look identical.

The trap is the one people reach for first: `list(set(xs))` is a one-liner that **destroys
order**, and the order it produces is not random but not meaningful either — it follows the hash
values. It looks stable in testing (small ints hash to themselves) and reorders in production
(strings are hash-randomised per process, NB-02 §1.4 and §3 below).

In [16]:
# ---------------------------------------------------------------------------
# 2.4 Deduplication: three versions, one of which quietly reorders.
# ---------------------------------------------------------------------------
def dedupe_list(xs):
    """The accidentally-quadratic version: `in` on a list is a linear scan."""
    out = []
    for x in xs:
        if x not in out:
            out.append(x)
    return out


def dedupe_set(xs):
    """Order-preserving and O(n): the set is only for membership."""
    seen = set()
    out = []
    for x in xs:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def dedupe_dict(xs):
    """Same thing, using dict's guaranteed insertion order (Python 3.7+)."""
    return list(dict.fromkeys(xs))


for name, fn in (("dedupe_set", dedupe_set), ("dedupe_dict", dedupe_dict)):
    checked = stress(fn, dedupe_list, lambda r: [r.randrange(-8, 8)
                                                 for _ in range(r.randrange(0, 30))],
                     n=3000, seed=RANDOM_SEED, label=name)
    print("%-13s %s random lists identical to the order-preserving reference"
          % (name + ":", "{:,}".format(checked)))

print()
words = ["pear", "apple", "fig", "apple", "pear", "date"]
print("input          :", words)
print("dedupe_set     :", dedupe_set(words), "  <- first-occurrence order")
print("list(set(...)) :", list(set(words)), "  <- hash order, and it changes between runs")

dedupe_set:   3,000 random lists identical to the order-preserving reference
dedupe_dict:  3,000 random lists identical to the order-preserving reference

input          : ['pear', 'apple', 'fig', 'apple', 'pear', 'date']
dedupe_set     : ['pear', 'apple', 'fig', 'date']   <- first-occurrence order
list(set(...)) : ['pear', 'fig', 'date', 'apple']   <- hash order, and it changes between runs


In [17]:
# ---------------------------------------------------------------------------
# The list-vs-set membership cost, measured.
# ---------------------------------------------------------------------------
def all_distinct(n):
    return list(range(n))                      # worst case: nothing to skip


print("`x not in out` on a list -- O(n^2):")
growth_table(measure_growth(dedupe_list, [1_000, 2_000, 4_000, 8_000],
                            setup=all_distinct, repeats=3), claim="O(n^2)")
print()
print("`x not in seen` on a set -- O(n):")
growth_table(measure_growth(dedupe_set, [200_000, 400_000, 800_000, 1_600_000],
                            setup=all_distinct, repeats=3), claim="O(n)")

`x not in out` on a list -- O(n^2):


         n        seconds      ratio
------------------------------------
     1,000       0.003883          -
     2,000       0.016806       4.33
     4,000       0.070011       4.17
     8,000       0.276423       3.95

best fit: O(n^2) (relative error 0.047); next: O(n log n) (1.275)
claimed O(n^2) -> measurement MATCHES the claim

`x not in seen` on a set -- O(n):


         n        seconds      ratio
------------------------------------
   200,000       0.023788          -
   400,000       0.047983       2.02
   800,000       0.098050       2.04
 1,600,000       0.196610       2.01

best fit: O(n) (relative error 0.014); next: O(n log n) (0.046)
claimed O(n) -> measurement MATCHES the claim


[('O(n)', 0.013896942185651469),
 ('O(n log n)', 0.04556168388911538),
 ('O(log n)', 1.2984789152820542),
 ('O(n^2)', 1.4194247122702217),
 ('O(1)', 1.5202140959521395),
 ('O(2^n)', 1.5202140959521395),
 ('O(n^3)', 10.045650658330574)]

The two functions differ by one word — `out` versus `seen` — and by a complexity class. This is
the single most common accidental quadratic in working code, because `if x not in collection`
reads identically whatever `collection` is, and Python will not warn you.

Rules worth carrying:

- **Membership test in a loop? The container must be a `set` or `dict`.** Not a `list`, not a
  `tuple`.
- **Need order preserved? `dict.fromkeys` or an explicit `seen` set.** Never `list(set(xs))`
  unless order genuinely does not matter — and say so in a comment, because the next reader
  cannot tell whether you decided that or forgot.
- **The elements must be hashable.** For lists of lists, convert to tuples first (§2.1).

***
# Part 3 - The signature difficulty: when O(1) becomes O(n)

Every previous section measured a hash table behaving well on *random* input. That is the right
model for most software and the wrong model for any service that accepts input from strangers,
because **the average case is an average over inputs, and an attacker chooses the input.**

The threat is concrete. A web server that parses query parameters, JSON keys, HTTP headers or form
fields into a dictionary is letting a remote party choose its hash-table keys. If that party can
find keys that collide, every insertion walks a chain of length $\Theta(n)$, building the
dictionary costs $\Theta(n^2)$, and a request that should take a millisecond takes minutes. This
is **hash flooding**, it was demonstrated against most major web stacks in 2011, and the defences
in CPython and the JVM today exist because of it.

Three questions, answered by measurement:

1. **How hard is it to construct colliding keys?** (§3.1 — for Java's `String.hashCode`, trivial.)
2. **What does it cost the victim?** (§3.2 — CPython's `dict` goes quadratic.)
3. **What do the defences actually defend?** (§3.3 — and each language leaves the other's hole open.)

## 3.1 Constructing collisions for a fixed hash function

Java specifies `String.hashCode` exactly, and has since 1.0:

$$ h = s_0 \cdot 31^{n-1} + s_1 \cdot 31^{n-2} + \dots + s_{n-1} $$

A specified, unkeyed, published function is one an attacker can run offline. And this one has a
gift: `"Aa"` and `"BB"` have the same hash. Since the hash of a concatenation is determined by the
hashes of fixed-length blocks, **any** string built from those two blocks collides with every
other — so $k$ blocks give $2^k$ colliding strings, generated in a loop, no search required.

This is not an obscure trick; it is the standard demonstration, and it works on any unkeyed
polynomial hash including §1.1's `poly31`.

In [18]:
# ---------------------------------------------------------------------------
# 3.1 2^k strings with one Java hashCode. Built, then verified in Java itself.
# ---------------------------------------------------------------------------
def java_string_hash(s):
    """Java's String.hashCode, in Python: 32-bit signed, base 31."""
    h = 0
    for ch in s:
        h = (31 * h + ord(ch)) & 0xFFFFFFFF
    return h - (1 << 32) if h >= (1 << 31) else h


def colliding_strings(count):
    """count strings that all share one Java hashCode."""
    bits = max(1, (count - 1).bit_length())
    return ["".join("BB" if (i >> b) & 1 else "Aa" for b in range(bits))
            for i in range(count)]


print("java_string_hash('Aa') =", java_string_hash("Aa"),
      "   java_string_hash('BB') =", java_string_hash("BB"))
print()
for k in (4, 8, 1024):
    fam = colliding_strings(k)
    print("  %5d generated keys -> %d distinct hashCode, %d distinct strings"
          % (len(fam), len(set(java_string_hash(s) for s in fam)), len(set(fam))))
print()
print("  first four:", colliding_strings(4))

# Cross-check the Python model against the real JVM.
CHECK_SRC = r"""
import java.util.*;
public class HashCheck {
    public static void main(String[] args) throws Exception {
        Scanner sc = new Scanner(System.in);
        System.out.println(sc.nextLine().hashCode());
    }
}
"""
if JAVA:
    checked = cross_check(java_string_hash, CHECK_SRC,
                          lambda r: "".join(r.choice("AaBb09 _") for _ in range(r.randint(1, 30))),
                          lambda s: s + "\n", n=60, seed=RANDOM_SEED,
                          label="java_string_hash")
    print()
    print("Python model vs the real JVM: %d random strings, identical hashCode." % checked)
else:
    print("\nJDK not available; the Python model was not cross-checked.")

java_string_hash('Aa') = 2112    java_string_hash('BB') = 2112

      4 generated keys -> 1 distinct hashCode, 4 distinct strings
      8 generated keys -> 1 distinct hashCode, 8 distinct strings
   1024 generated keys -> 1 distinct hashCode, 1024 distinct strings

  first four: ['AaAa', 'BBAa', 'AaBB', 'BBBB']



Python model vs the real JVM: 60 random strings, identical hashCode.


1,024 keys, one hash value. The Python model of `String.hashCode` agrees with the actual JVM on
random strings, so the collisions are real rather than an artefact of the model.

Python is not vulnerable to *this* family — but for a reason that has nothing to do with the
family itself, as §3.3 shows. First, the damage.

One measurement note that applies to the rest of this part. Every timing below builds its keys in
`setup`, whose cost `measure_growth` excludes, so only the dictionary insertion is timed. Building
the keys inside the timed function instead makes the colliding family look 10× slower in Python
purely because those keys are longer strings — a mistake worth naming, because it would have
supported a conclusion that is the opposite of the truth.

## 3.2 What it costs the victim: CPython's `dict`, made quadratic

To attack CPython we need keys colliding under *its* hash, and `str` is protected (§3.3). But
`hash()` on an `int` is not a mixing function at all:

- for small integers, `hash(i) == i`;
- in general, `hash(i) == i mod (2^61 - 1)`.

That second line is the whole attack. $2^{61}-1$ is a Mersenne prime, `hash(2**61 - 1) == 0`, and
therefore $i$ and $i + (2^{61}-1)$ have **identical hashes**. Generating thousands of colliding
integer keys takes one multiplication each.

And integer keys are not exotic. User IDs, order numbers, timestamps, product SKUs — any of these
arriving from outside and being used as dictionary keys is this attack's target.

In [19]:
# ---------------------------------------------------------------------------
# 3.2 Integers with identical CPython hashes.
# ---------------------------------------------------------------------------
P = 2 ** 61 - 1          # CPython reduces int hashes modulo this Mersenne prime

print("hash(2**61 - 1) =", hash(P))
print("hash(1) = %d,  hash(1 + P) = %d,  hash(1 + 2P) = %d"
      % (hash(1), hash(1 + P), hash(1 + 2 * P)))
print("all equal:", hash(1) == hash(1 + P) == hash(1 + 2 * P),
      "  but the keys are distinct:", len({1, 1 + P, 1 + 2 * P}))
print()


def build_dict(keys):
    """Only this is timed. The keys are built in setup, so we measure insertion."""
    d = {}
    for k in keys:
        d[k] = 1
    return d


def colliding_ints(n):
    return [1 + i * P for i in range(n)]


def ordinary_ints(n):
    return list(range(n))


print("  %8s %16s %16s %10s" % ("n", "colliding (s)", "ordinary (s)", "ratio"))
print("  " + "-" * 54)
for n in (2_000, 4_000, 8_000, 16_000):
    tb = measure_growth(build_dict, [n], setup=colliding_ints, repeats=3)[0]["seconds"]
    tg = measure_growth(build_dict, [n], setup=ordinary_ints, repeats=7)[0]["seconds"]
    print("  %8s %16.5f %16.6f %9.0fx" % ("{:,}".format(n), tb, tg, tb / tg))

hash(2**61 - 1) = 0
hash(1) = 1,  hash(1 + P) = 1,  hash(1 + 2P) = 1
all equal: True   but the keys are distinct: 3

         n    colliding (s)     ordinary (s)      ratio
  ------------------------------------------------------
     2,000          0.02841         0.000119       239x


     4,000          0.12444         0.000159       784x


     8,000          0.50800         0.000322      1580x


    16,000          2.09831         0.000627      3349x


In [20]:
# ---------------------------------------------------------------------------
# The ratio is growing. That means a complexity difference, so measure it.
# ---------------------------------------------------------------------------
print("dict insertion, keys with IDENTICAL hashes:")
growth_table(measure_growth(build_dict, [2_000, 4_000, 8_000, 16_000],
                            setup=colliding_ints, repeats=3), claim="O(n^2)")
print()
print("dict insertion, ordinary keys:")
growth_table(measure_growth(build_dict, [200_000, 400_000, 800_000, 1_600_000],
                            setup=ordinary_ints, repeats=3), claim="O(n)")

dict insertion, keys with IDENTICAL hashes:


         n        seconds      ratio
------------------------------------
     2,000       0.027761          -
     4,000       0.122968       4.43
     8,000       0.508252       4.13
    16,000       2.089789       4.11

best fit: O(n^2) (relative error 0.063); next: O(n^3) (1.271)
claimed O(n^2) -> measurement MATCHES the claim

dict insertion, ordinary keys:


         n        seconds      ratio
------------------------------------
   200,000       0.016026          -
   400,000       0.033980       2.12
   800,000       0.067406       1.98
 1,600,000       0.129546       1.92

best fit: O(n) (relative error 0.025); next: O(n log n) (0.062)
claimed O(n) -> measurement MATCHES the claim


[('O(n)', 0.025035381369684792),
 ('O(n log n)', 0.06236296420389687),
 ('O(log n)', 1.287499859291868),
 ('O(n^2)', 1.4827360837386396),
 ('O(1)', 1.5070918325937923),
 ('O(2^n)', 1.5070918325937923),
 ('O(n^3)', 10.374085532750502)]

**$O(n)$ becomes $O(n^2)$, and the ratio grows 4× per doubling** — the signature of a complexity
difference rather than a constant factor, exactly as in NB-02 §3.

The absolute numbers are what make it an attack rather than a curiosity. At n = 16,000 the
colliding build takes about **two seconds** against under a millisecond for ordinary keys, and
because it is quadratic, 160,000 keys would take about 200 seconds. A single HTTP request carrying
a few hundred kilobytes of crafted JSON can occupy a CPU core for minutes. That is a denial of
service from one request, with no volume of traffic required.

Note also what the table is *not* doing wrong: all 16,000 keys are stored, correctly and
retrievably. Nothing is broken. The structure is behaving exactly as designed, on input chosen to
make the design's worst case the common case.

## 3.3 What the defences actually defend

Both runtimes were patched after the 2011 disclosures, and they chose **different** defences —
which is why each is exposed where the other is not.

**CPython: randomise the hash function.** Since 3.3, `str` and `bytes` are hashed with **SipHash**
keyed by a per-process random seed, so an attacker cannot precompute colliding strings without
knowing the seed. Set `PYTHONHASHSEED=0` to disable it and the old attacks work again.

**The JVM: fix the data structure instead.** `String.hashCode` is specified and cannot change
without breaking serialisation and every hash-order-dependent program in existence. So Java 8 made
`HashMap` convert an over-long bucket into a **red-black tree** once it holds 8 entries and the
table has at least 64 buckets. Collisions still happen; they just cost $O(\log n)$ instead of
$O(n)$.

Each defence has the other's hole:

| | CPython `dict` | Java `HashMap` |
|---|---|---|
| **String keys** | protected — collisions cannot be precomputed | wide open — §3.1 built 1,024 in a loop |
| **Integer keys** | wide open — `hash(i) == i` | same exposure in principle |
| **Once you do collide** | $O(n)$ per bucket → $O(n^2)$ total | $O(\log n)$ per bucket → treeified |

Both halves are measurable.

In [21]:
# ---------------------------------------------------------------------------
# 3.3a Does the Java-colliding family hurt Python? And is str hashing random?
# ---------------------------------------------------------------------------
fam = colliding_strings(8)
print("The 8 strings that share ONE Java hashCode, hashed by Python:")
print("  distinct Python hashes:", len(set(hash(s) for s in fam)), "out of", len(fam))
print()


def control_strings(n):
    """Ordinary keys of EXACTLY the same length as the colliding ones.

    Without this the comparison is unfair: 2^k colliding keys need k blocks, so
    they are far longer than "key123" and cost more to hash for a reason that
    has nothing to do with collisions.
    """
    width = len(colliding_strings(n)[0])
    return ["%0*d" % (width, i) for i in range(n)]


print("  %8s %18s %18s %10s" % ("n", "Aa/BB keys (s)", "same-length (s)", "ratio"))
print("  " + "-" * 58)
for n in (8_000, 16_000, 32_000, 64_000):
    tb = measure_growth(build_dict, [n], setup=colliding_strings, repeats=5)[0]["seconds"]
    tg = measure_growth(build_dict, [n], setup=control_strings, repeats=5)[0]["seconds"]
    print("  %8s %18.5f %18.5f %9.2fx" % ("{:,}".format(n), tb, tg, tb / tg))

print()
print("A ratio of about 1.0: Python's SipHash does not care that these strings")
print("were constructed to collide under a different hash function. The keys")
print("that cost Java 10x cost Python nothing.")

The 8 strings that share ONE Java hashCode, hashed by Python:
  distinct Python hashes: 8 out of 8

         n     Aa/BB keys (s)    same-length (s)      ratio
  ----------------------------------------------------------
     8,000            0.00065            0.00064      1.02x


    16,000            0.00126            0.00126      1.00x


    32,000            0.00275            0.00277      1.00x


    64,000            0.00736            0.00750      0.98x

A ratio of about 1.0: Python's SipHash does not care that these strings
were constructed to collide under a different hash function. The keys
that cost Java 10x cost Python nothing.


In [22]:
# ---------------------------------------------------------------------------
# 3.3b Randomisation is per PROCESS, and covers str/bytes only.
# ---------------------------------------------------------------------------
import subprocess

print("hash('hello') in three fresh interpreters:")
for _ in range(3):
    out = subprocess.run([sys.executable, "-c",
                          "print(hash('hello'), hash(42))"],
                         capture_output=True, text=True, check=True).stdout.split()
    print("  hash('hello') = %-22s hash(42) = %s" % (out[0], out[1]))

print()
print("The string hash changes every run; the integer hash never does.")
print("That is the shape of the defence: SipHash protects str and bytes.")
print("int, float and tuple-of-ints hashes are fixed, published, and were")
print("exactly what section 3.2 exploited.")

hash('hello') in three fresh interpreters:
  hash('hello') = 4764371943242180929    hash(42) = 42
  hash('hello') = -6913943408487142310   hash(42) = 42
  hash('hello') = -513535316061259371    hash(42) = 42

The string hash changes every run; the integer hash never does.
That is the shape of the defence: SipHash protects str and bytes.
int, float and tuple-of-ints hashes are fixed, published, and were
exactly what section 3.2 exploited.


In [23]:
# ---------------------------------------------------------------------------
# 3.3c The other half: Java under the attack Python could not survive.
# ---------------------------------------------------------------------------
TREEIFY_SRC = r"""
import java.util.*;

public class Treeify {
    static List<String> colliding(int count) {
        List<String> out = new ArrayList<>();
        int bits = 32 - Integer.numberOfLeadingZeros(Math.max(count - 1, 1));
        for (int i = 0; i < count; i++) {
            StringBuilder sb = new StringBuilder();
            for (int b = 0; b < bits; b++) sb.append(((i >> b) & 1) == 0 ? "Aa" : "BB");
            out.add(sb.toString());
        }
        return out;
    }
    /** Ordinary keys of exactly the same LENGTH as the colliding ones, so the
     *  comparison isolates collisions rather than string length. */
    static List<String> ordinary(int count) {
        int width = colliding(count).get(0).length();
        List<String> out = new ArrayList<>();
        for (int i = 0; i < count; i++) {
            String s = Integer.toString(i);
            StringBuilder sb = new StringBuilder();
            while (sb.length() < width - s.length()) sb.append('0');
            out.add(sb.append(s).toString());
        }
        return out;
    }
    /** Build the map and look every key up. Best of `reps`, in nanoseconds. */
    static long bench(List<String> keys, int reps) {
        long best = Long.MAX_VALUE;
        for (int r = 0; r < reps; r++) {
            long t0 = System.nanoTime();
            Map<String,Integer> m = new HashMap<>();
            for (String k : keys) m.put(k, 1);
            int found = 0;
            for (String k : keys) if (m.get(k) != null) found++;
            long dt = System.nanoTime() - t0;
            if (found != keys.size()) throw new IllegalStateException("lost keys");
            best = Math.min(best, dt);
        }
        return best;
    }
    public static void main(String[] args) {
        for (int w = 0; w < 3; w++) {          // let the JIT settle before timing
            bench(colliding(4000), 2);
            bench(ordinary(4000), 2);
        }
        System.out.printf("  %8s %18s %18s %8s%n",
                          "n", "colliding (ms)", "same-length (ms)", "ratio");
        System.out.println("  " + "-".repeat(56));
        for (int n : new int[]{8000, 16000, 32000, 64000}) {
            double c = bench(colliding(n), 5) / 1e6;
            double o = bench(ordinary(n), 5) / 1e6;
            System.out.printf("  %8d %18.2f %18.2f %7.1fx%n", n, c, o, c / o);
        }
    }
}
"""

if JAVA:
    print("java.util.HashMap, every key colliding (the attack from 3.1):")
    print(run_java(TREEIFY_SRC))
else:
    print("JDK not available; skipping.")

java.util.HashMap, every key colliding (the attack from 3.1):


         n     colliding (ms)   same-length (ms)    ratio
  --------------------------------------------------------
      8000               3.61               0.32    11.5x
     16000               6.50               0.54    12.0x
     32000              14.33               1.51     9.5x
     64000              33.67               3.10    10.9x



Read the three ratio columns of this part against each other.

| Attack | Victim | Slowdown as n goes 2k→16k | Verdict |
|---|---|---|---|
| Colliding **ints** (§3.2) | CPython `dict` | hundreds × → **thousands ×**, quadrupling | unbounded, $O(n^2)$ |
| Colliding **strings** (§3.1) | CPython `dict` | **≈ 1.0×** throughout | no effect |
| Colliding **strings** (§3.1) | Java `HashMap` | **≈ 10×**, roughly flat | bounded |

The Python integer ratio **keeps multiplying** as n grows; the Java string ratio **holds around
10×**. (Exact figures wander between runs — these are wall-clock timings — but the shapes do not.) That is the
whole difference between an outage and a slowdown. Java is doing about ten times more work per key
than it should, which is not nothing — but it is a constant overhead on a worse structure, not a
change of complexity class. The bucket became a red-black tree, so lookups cost $O(\log n)$ and
the attack degrades the service instead of destroying it.

Both Java columns use keys of **identical length**, which matters: $2^k$ colliding keys need $k$
blocks, so they are far longer than `"key123"` and cost more to hash for reasons unrelated to
collisions. Comparing against a same-length control is what makes the 10× attributable to the
collisions rather than to string length.

One real caveat remains: **treeification needs the keys to be `Comparable`.** `String` is. A
custom key class that is not falls back to linear chains, and Java is then exactly as exposed as
Python.

**What to actually do.** The runtime defence is a backstop, not a plan:

1. **Do not build unbounded dictionaries from untrusted input.** Cap the number of parameters,
   JSON keys and headers you will parse. This is the real fix and every mature web framework does it.
2. **Never disable hash randomisation** — no `PYTHONHASHSEED=0` in production. It is occasionally
   set to make test output reproducible; that habit must not follow the code to a server.
3. **Watch integer and tuple keys**, which no runtime randomises.
4. **For your own hash functions on untrusted input, key them** — a random per-process seed, as
   §3.1 of NB-02 recommended for Rabin-Karp's base. Same threat, same fix.
5. **Prefer a structure with worst-case guarantees where an adversary is in scope.** A balanced
   tree (NB-08) is $O(\log n)$ *guaranteed*, which is sometimes worth more than $O(1)$ expected.

***
# Part 4 - Tough questions

***

### Q1. Are hash tables O(1)? Answer carefully.

<details><summary>Answer</summary>

**They are $\Theta(1)$ expected, amortised, under simple uniform hashing. Every one of those three
qualifiers is load-bearing, and the worst case is $\Theta(n)$.**

- **Expected**, not worst-case. A single lookup can touch every element if they all collide (§3).
- **Amortised**, because a `put` that triggers a resize is $\Theta(n)$; doubling spreads that over
  $n$ insertions (§1.3, the same argument as NB-01's dynamic array).
- **Under uniform hashing** — the assumption that keys spread evenly. §1.1 shows a bad hash breaks
  it and §3 shows an *adversary* breaks it on purpose.

The honest one-line answer in an interview: "$O(1)$ expected, $O(n)$ worst case, and whether the
worst case is reachable depends on whether an attacker picks the keys." That last clause is what
separates someone who has used hash tables from someone who understands them.

The comparison that makes it concrete: a balanced BST (NB-08) is $O(\log n)$ *guaranteed*. For
$n = 10^6$ that is about 20 operations, always. A hash table is ~1 operation typically and
$10^6$ in the worst case. Which you want depends entirely on whether "typically" is a promise you
can rely on.

</details>

***

### Q2. Chaining or open addressing — how do you choose?

<details><summary>Answer</summary>

| | Chaining | Open addressing |
|---|---|---|
| Load factor | works at $\alpha > 1$ | must keep $\alpha < 1$, in practice $< 0.75$ |
| Memory | pointer + node per entry | one flat array, no per-entry overhead |
| Cache | a pointer chase per probe | contiguous, cache-friendly (NB-01 §3) |
| Delete | trivial: unlink | needs tombstones (§1.4) |
| Degradation | graceful, linear in $\alpha$ | cliff-edged, $(1-\alpha)^{-2}$ |

**Open addressing wins on cache and memory at low load**, which is why CPython's `dict`, Java's
`IdentityHashMap`, and most high-performance tables use it. Its costs are the delete complexity
and the resize discipline: you *must* keep $\alpha$ down and you *must* handle tombstones, or §1.4
and §1.5 come for you.

**Chaining wins on simplicity and robustness.** Deletes are trivial, it tolerates $\alpha > 1$,
and its worst case is a merely-long list rather than a poisoned probe sequence — which is why
Java's `HashMap` chains (then treeifies, §3.3). If you are not sure, chain; it has fewer ways to
go quietly wrong.

The measured reason open addressing needs a lower ceiling is §1.5: at $\alpha = 0.95$ probing
costs ~180 probes to chaining's ~1. The formulas *predict* the ceiling.

</details>

***

### Q3. Why must the table size and the growth policy be chosen with care?

<details><summary>Answer</summary>

Two independent decisions, each with a classic failure.

**Table size and the modulo.** Reducing a hash with `h % m` keeps only `h mod m`. If `m` shares
structure with your hashes, you lose the good bits:

- **Power-of-two $m$** keeps only the low bits of the hash — fast (`h & (m-1)`), but lethal if the
  low bits are not well mixed. §3.2's integer attack works *because* `hash(i) == i` and the low
  bits are just `i`. CPython uses power-of-two tables and defends by perturbing the probe sequence
  with the high bits of the hash.
- **Prime $m$** mixes all the bits into the result and is the classic advice, at the cost of a
  slower modulo than a mask.

**Growth policy.** Grow by a constant *factor* (double), never a constant *amount*. Growing by a
fixed 100 slots makes $n$ insertions cost $\Theta(n^2)$ — the same trap as NB-01 §2.2's dynamic
array, for the same reason. Doubling gives $\Theta(1)$ amortised.

The subtle one: **you resize on the used-slot count, tombstones included** (§1.4), not just the
live count. A table that is mostly tombstones is "empty" by live count and slow by probe count, so
open-addressed tables rebuild when tombstones pile up, not only when they fill.

</details>

***

### Q4. Walk through deleting from an open-addressed table.

<details><summary>Answer</summary>

**You cannot just empty the slot.** An empty slot is a signal — it tells every lookup "stop, the
key is not here", because a lookup walks the probe sequence and halts at the first empty slot. If
you empty a slot that some other key probed *through*, you cut that key off from its home, and it
becomes unreachable while still sitting in the table.

§1.4 measured exactly this: `put -10, put 14, del -10` leaves `14` in slot 7 but unreachable,
because clearing slot 6 makes `14`'s probe from its home slot 6 stop immediately at empty.

**The fix is a tombstone**: a third slot state, distinct from empty and occupied, meaning "keep
probing past me". Lookups treat it as occupied (do not stop); inserts may reuse it (once they have
confirmed the key is absent — you must scan to an empty slot *first*, or you can insert a duplicate).

**The cost:** tombstones accumulate and slow every probe, because they count toward the effective
load factor even though they hold no data. So the table tracks used-slots-including-tombstones and
rebuilds when that crosses the threshold — dropping the tombstones in the rehash (§1.4's `_resize`
does exactly this).

This is *the* reason many engineers reach for chaining: its delete is `list.remove`, no third
state, no accounting. The complexity of open-addressed delete is a real cost, weighed against its
real cache-locality benefit.

</details>

***

### Q5. Explain the load factor and the numbers 0.75 and 0.66.

<details><summary>Answer</summary>

The **load factor** $\alpha = n/m$ is entries over slots, and it is the one knob that trades
memory against speed. Low $\alpha$: fast, wasteful. High $\alpha$: dense, slow.

The specific thresholds come straight from §1.5's formulas:

- **Java `HashMap` resizes at $\alpha = 0.75$.** It chains, so its cost grows linearly in $\alpha$;
  0.75 is a comfortable balance of memory against a still-short average chain, and it treeifies to
  cap the tail.
- **CPython `dict` resizes at $\alpha \approx 0.66$ (2/3).** It uses open addressing, whose cost
  grows as $(1-\alpha)^{-2}$, so it must stay lower — at 2/3 the expected probe count is about 2.5,
  and past there the curve steepens fast.

The different numbers are not arbitrary taste; they are the two different curves in §1.5 evaluated
at "still cheap". Chaining can afford a higher ceiling because its curve is gentler.

Raising the factor saves memory and costs time on every lookup; lowering it does the reverse.
Resizing is $\Theta(n)$ but amortises to $\Theta(1)$ per insert. If you know your final size,
**presize the table** — construct it big enough once and skip the intermediate resizes entirely,
which for a large known dataset is a measurable win.

</details>

***

### Q6. Hash flooding: what is it, and what stops it?

<details><summary>Answer</summary>

**The attack.** A hash table degrades to $\Theta(n)$ per operation when keys collide, so building
one from $n$ colliding keys costs $\Theta(n^2)$ (§3.2, measured: ~2,000× at n = 16,000, growing).
Any service that turns untrusted input into hash-table keys — JSON, query strings, HTTP headers,
form fields — lets a remote attacker choose those keys. One crafted request can pin a CPU core for
minutes. Demonstrated against PHP, Java, Python, Ruby, Node and others in 2011 (the Klink–Wälde
"Effective DoS" work).

**The two defences**, and neither is complete on its own (§3.3):

1. **Randomise the hash (CPython).** `str`/`bytes` use SipHash with a per-process random key, so an
   attacker cannot precompute colliding strings. Does nothing for `int`, `float` or tuple keys,
   whose hashes are fixed — which is why §3.2's integer attack still works.
2. **Fix the structure (Java).** `HashMap` treeifies over-long buckets, so collisions cost
   $O(\log n)$ instead of $O(n)$. Works regardless of key type, but caps the damage rather than
   preventing it, and only for `Comparable` keys.

**What you actually do:** cap how much untrusted input you parse into a dictionary (the real
fix, and what every mature framework does); never set `PYTHONHASHSEED=0` in production; be aware
integer/tuple keys are unprotected; and where an adversary is genuinely in scope, consider a
structure with a worst-case guarantee (a balanced tree) instead of an expected one.

The through-line from NB-02 §2.4: an asymptotic worst case tells you what an *adversary* can do,
not what your *data* will do. When the adversary picks the input, the worst case is the case.

</details>

***

### Q7. What is the `hashCode`/`equals` contract, and what breaks it?

<details><summary>Answer</summary>

**The contract:** if `a.equals(b)` then `a.hashCode() == b.hashCode()`. The converse is not
required — unequal objects may share a hash (that is just a collision). Only the forward direction
matters, because a container uses the hash to pick the bucket and `equals` to confirm the hit;
break the implication and a key is filed under one hash and sought under another.

**Three ways to break it** (all measured in §1.6):

1. **Override `equals`, forget `hashCode`.** Equal objects hash differently → two map entries for
   one logical key, and lookups miss. `javac -Xlint:all` *refuses to compile this* under
   `-Werror`; you have to actively suppress the warning to ship it.
2. **Mutate a key after insertion.** The hash was right when stored and is now wrong → the entry
   is stranded: unreachable by lookup, still counted by `size()`, still holding a GC reference.
   §1.6 showed this fires *intermittently* in Python too, depending on whether the new hash happens
   to probe the same slot.
3. **`hashCode` and `equals` disagree on which fields they use.** Same failure as (1) by a subtler
   route — `equals` compares three fields, `hashCode` hashes two, so objects equal on the hashed
   fields but differing on the third collide-and-mismatch forever.

**The universal defence: immutable keys.** Java `record`s and `final` fields; Python enforces it
by making `list`/`dict`/`set` unhashable and setting `__hash__ = None` when you define `__eq__`
without it — a `TypeError` at insertion beats a silent misfile.

</details>

***

### Q8. Sets and dicts are the "same" structure — when does the distinction matter?

<details><summary>Answer</summary>

A `set` is a hash table storing only keys; a `dict` stores keys with values. Same collision
handling, same load-factor behaviour, same worst case. Choosing between them is about intent, and
the intent shows up as bugs when it is wrong:

- **Membership, no data → `set`.** §2.4: `x in seen` is the whole point. Using a dict with dummy
  values works but signals the wrong thing to the reader.
- **Association → `dict`.** Key to value.
- **Counting → `dict` / `Counter`.** §2.3's `counts.get(x, 0) + 1` turns a dict into a multiset,
  which a plain set cannot express.
- **Ordered iteration matters → `dict`** (insertion order, guaranteed since 3.7) **not `set`**
  (no order guarantee). §2.4's `list(set(xs))` bug is exactly this confusion: a set has no order,
  so deduping through one scrambles the sequence, invisibly in tests (small ints hash to
  themselves) and visibly in production (string hashes are randomised).

The trap that costs real time is none of these — it is using a **list** where a set belongs.
`if x in some_list` inside a loop is the most common accidental $O(n^2)$ in working code (§2.4,
measured), and it reads identically to the $O(n)$ version. In review, every `in` inside a loop
deserves a glance at what is on the right of it.

</details>

***

### Q9. How does CPython's `dict` actually work?

<details><summary>Answer</summary>

Worth knowing because it explains §3.2 and because it is a well-engineered real table.

- **Open addressing**, not chaining — for the cache locality of §1.5's discussion.
- **Power-of-two size**, so the modulo is a bitmask `h & (m-1)`. This keeps only the low bits, so
  on a collision the probe sequence is *perturbed by the high bits* of the hash (the `perturb`
  shift in `dictobject.c`), which recovers some mixing the mask threw away. It does not save you
  from §3.2, where the whole hash — high bits included — is engineered to collide.
- **Split keys/values ("compact dict", 3.6+):** entries live in a dense insertion-ordered array
  and the hash table holds indices into it. This is *why* dicts preserve insertion order (§2.4),
  and it cut dict memory by roughly a third.
- **Resizes at $\alpha \approx 2/3$** (Q5), growing 4× for small dicts and 2× for large ones.
- **`str`/`bytes` hashed with keyed SipHash** (§3.3); `int` hashed as `i mod (2^61−1)` (§3.2).

The one-sentence version: a compact, insertion-ordered, open-addressed, power-of-two table with a
perturbed probe sequence and a randomised string hash. Each adjective is a decision this notebook
measured the reason for.

</details>

***

### Q10. When is a hash table the wrong choice?

<details><summary>Answer</summary>

Its strengths are exactly its weaknesses inverted, so §"what it is bad at" is the answer:

- **You need order.** Hash tables have none (a dict's *insertion* order is not *sorted* order). For
  min/max, successor/predecessor, or sorted iteration, use a balanced BST (NB-08) or a heap
  (NB-09). Getting sorted output from a hash table means an $O(n \log n)$ sort afterwards.
- **You need range queries.** "All keys between 10 and 20" is $\Theta(n)$ in a hash table — you
  must scan everything — and $O(\log n + k)$ in a balanced tree. Hashing deliberately destroys the
  locality a range query needs.
- **You need worst-case guarantees.** §3: adversarial input makes it $\Theta(n)$. Real-time systems
  and adversary-facing services often prefer a tree's guaranteed $O(\log n)$.
- **Keys are not hashable, or hashing is expensive.** Mutable keys are out (§1.6). If keys are huge
  and hashing means reading all of each one, the constant factor can erase the benefit.
- **Tiny $n$.** For a handful of entries a linear scan of an array beats hashing's constant
  factor and its memory overhead. Hash tables carry real per-entry and empty-slot overhead — a
  table at $\alpha = 0.5$ is half empty by design.

The meta-answer: a hash table trades ordering, range queries and worst-case guarantees for
expected-$O(1)$ point access. When you need any of the things it traded away, it is the wrong tool.

</details>

***

### Q11. `hash(-1)`, `hash(1.0) == hash(1)`, `hash(True)` — explain.

<details><summary>Answer</summary>

Small edge cases that reveal how Python's hashing is designed, and each has bitten someone.

- **`hash(-1) == -2`.** `-1` is CPython's C-level "error occurred" sentinel for `hash()`, so the
  one integer that would naturally hash to `-1` is remapped to `-2`. A curiosity — unless you are
  writing a `__hash__` in a C extension, where returning `-1` silently means "error".
- **`hash(1.0) == hash(1) == hash(1 + 0j) == 1`.** Deliberate: objects that compare equal *must*
  hash equal (Q7's contract), and `1 == 1.0 == 1+0j` is `True`, so `{1: "a", 1.0: "b"}` is one
  entry with value `"b"`. Python defines a single hash across all numeric types to honour this.
  It surprises people who expect the `int` and the `float` to be different keys.
- **`hash(True) == hash(1) == 1`, `hash(False) == 0`.** `bool` subclasses `int` with
  `True == 1`, so `{1: "a", True: "b"}` collapses to one entry. This genuinely bites: a dict keyed
  on mixed `1`/`True` or `0`/`False` silently merges them.

The unifying rule is Q7's contract taken seriously across *all* types: **equal values hash equal,
always**, even across types that look unrelated. Everything above is that rule's consequences.

</details>

***

## Coding challenges

### Challenge 1 — build a treeifying HashMap

§3.3 said Java caps collision damage by turning long buckets into trees. Build it.

1. Start from §1.3's `ChainingMap`. When any bucket exceeds a threshold (Java uses 8), convert
   *that bucket* from a list to a balanced BST keyed on the hash then the key. Convert back below
   a lower threshold (Java uses 6 — the gap is hysteresis, to avoid flapping).
2. Reproduce §3.2's attack against both the plain and the treeifying version, and measure. Plain
   should be $O(n^2)$; treeifying should be about $O(n \log n)$.
3. Your keys must be orderable for the tree. Handle the case where they are not — what does Java
   do? (Answer: falls back to a linear chain. Implement that fallback and show it is $O(n^2)$
   again, which is the caveat §3.3 flagged.)

### Challenge 2 — break your own hash function

1. Implement §1.1's `poly31`. Then, treating it as an attacker, construct 1,000 strings that all
   collide under it — the `"Aa"`/`"BB"` trick generalises to any unkeyed polynomial hash; find two
   two-character strings that collide and build from them.
2. Feed them to a table using `poly31` and measure the degradation.
3. Now key the hash: mix in a per-process random salt. Show your construction fails against the
   keyed version. You have just reinvented the CPython 3.3 defence — and NB-02 §2.4's advice for
   Rabin-Karp, which is the same fix for the same reason.

### Challenge 3 — a perfect hash

For a *static* key set (known in advance, never changed) you can do better than expected $O(1)$:
**guaranteed** $O(1)$ with no collisions at all.

1. Implement **FKS two-level hashing**: a top hash into buckets, and for each bucket a second hash
   sized $O(b^2)$ where $b$ is the bucket's occupancy — big enough that the birthday bound (§1.2)
   makes a collision-free second hash findable by trial in $O(1)$ expected attempts.
2. Verify zero collisions and total space $O(n)$.
3. Explain why this is useless for a dynamic table (adding a key can force a full rebuild) and
   where it is used anyway: compilers hashing language keywords, `gperf`, read-only lookup tables.

***
# Part 5 - Practice

| # | Exercise | The pattern | Difficulty |
|---|---|---|---|
| 1 | First non-repeating character | Counting map + order | ★☆☆☆☆ |
| 2 | Ransom note | Multiset containment | ★☆☆☆☆ |
| 3 | Longest consecutive sequence | Set membership, not sorting | ★★★☆☆ |
| 4 | LRU cache | Hash map + doubly linked list | ★★★★☆ |
| 5 | Subarray sum equals k, variants | Prefix sums in a map | ★★★☆☆ |
| 6 | Isomorphic strings | Two-way mapping | ★★☆☆☆ |
| 7 | 4-sum II | Split the search space | ★★★☆☆ |
| 8 | Insert/Delete/GetRandom in O(1) | Map + array, together | ★★★★☆ |

***

### 1. First non-repeating character

Return the first character in a string that appears exactly once.

- **Brief:** one pass to count (§2.3's counting-map pattern), a second pass in string order to
  find the first with count 1. State why two passes and not one.
- **Good result:** $O(n)$, and correct on the empty string and on all-repeating input.
- **The trap:** iterating a plain dict for "first" relies on insertion order — correct in
  Python 3.7+, a bug before it and in any language without the guarantee. Say what you are relying
  on. And define "character" (NB-02 §1.4) if the input is Unicode.

### 2. Ransom note

Can string `note` be built from the letters of string `magazine`, each letter used once?

- **Brief:** this is multiset containment. Count `magazine`, then decrement for `note`; fail if any
  count goes negative.
- **Good result:** $O(|note| + |magazine|)$, one map.
- **The trap:** the direction. You are asking whether `note`'s counts are all $\le$ `magazine`'s,
  not equal. Anagram (NB-02 §5) checks equality; this checks containment.

### 3. Longest consecutive sequence

Given an unsorted array, find the length of the longest run of consecutive integers (e.g.
`[100,4,200,1,3,2]` → 4, the run `1..4`). Required: $O(n)$.

- **Brief:** put everything in a set. For each value that is the **start** of a run (i.e. `x-1` is
  not in the set), walk `x, x+1, x+2, ...` counting. The start check is what keeps it linear.
- **Good result:** $O(n)$ despite the nested-looking loop — each value is walked at most once, an
  amortised argument like NB-01 §2.3's window.
- **The trap:** sorting is the obvious $O(n \log n)$ answer and the interviewer wants the set
  solution. Without the "is this a run start?" test the inner walk revisits elements and you are
  back to $O(n^2)$ — verify the linearity by counting inner iterations, do not assume it.

### 4. LRU cache

`get(key)` and `put(key, value)`, both $O(1)$, evicting the least-recently-used entry at capacity.

- **Brief:** a hash map for $O(1)$ lookup **plus** a doubly linked list (NB-04) for $O(1)$
  recency reordering. The map's values are list nodes. Neither structure alone can do it — this is
  the canonical "two structures, welded" problem.
- **Good result:** every operation genuinely $O(1)$; verify with a differential test against an
  $O(n)$ reference (an ordered list) over random operation sequences, exactly as §1.3 tested the
  maps.
- **The trap:** the linked-list pointer surgery on eviction and on move-to-front. Off-by-one on
  the head/tail sentinels is the classic bug. (Python's `OrderedDict.move_to_end` does this for
  you; implement it once by hand first, then use the built-in.)

### 5. Subarray sum equals k, and its variants

Extend §2.3: (a) longest subarray summing to k; (b) count subarrays with sum divisible by k;
(c) contiguous subarray with equal 0s and 1s.

- **Brief:** all three are the prefix-sum-in-a-map pattern with a twist. (a) stores the *earliest*
  index per prefix; (b) keys on `prefix mod k`; (c) maps 0→−1 and looks for equal prefixes.
- **Good result:** each $O(n)$, each verified against an $O(n^2)$ brute force.
- **The trap:** (b)'s negative-modulo (`((p % k) + k) % k` in most languages — NB-00 warned about
  Python vs Java `%` on negatives, and this is where it bites), and the `{0: ...}` seed from §2.3
  in every one of them.

### 6. Isomorphic strings

Do `s` and `t` have a consistent one-to-one character mapping (`"egg"`/`"add"` yes,
`"foo"`/`"bar"` no)?

- **Brief:** one map is not enough — `"badc"`/`"baba"` needs a check in **both** directions, or two
  characters map to one.
- **Good result:** $O(n)$, two maps (or a map plus a set of used targets).
- **The trap:** forgetting the reverse direction. It passes the obvious tests and fails on the
  many-to-one case. Build the counterexample first, then the code.

### 7. 4-sum II

Given four lists, count tuples `(i,j,k,l)` with `A[i]+B[i]+C[k]+D[l] == 0`.

- **Brief:** $O(n^4)$ is hopeless. Hash all sums `A[i]+B[j]` into a counting map, then for each
  `C[k]+D[l]` look up its negation. §2.2's complement lookup, split across two halves.
- **Good result:** $O(n^2)$ time and space, verified against the $O(n^4)$ brute force on small
  inputs.
- **The trap:** it is a *count*, so the map stores multiplicities (§2.3), and you add the looked-up
  count, not 1.

### 8. Insert / Delete / GetRandom, all O(1)

A container with `insert(x)`, `delete(x)` and `getRandom()`, every operation $O(1)$ expected.

- **Brief:** a hash map (value → index) **and** a dynamic array (NB-01), together. The array gives
  $O(1)$ random access; the map gives $O(1)$ delete by swapping the doomed element with the last
  and popping.
- **Good result:** all three $O(1)$; `getRandom` uniform. Differential-test insert/delete against a
  set, and check `getRandom`'s uniformity with a $\chi^2$ test (§1.1).
- **The trap:** delete is the whole exercise — the swap-with-last must also fix the moved element's
  index in the map. Do it in the wrong order and the map points at a popped slot.

***
# Part 6 - Reading

## Start here

**1. *Introduction to Algorithms* (CLRS), chapter 11 — "Hash Tables".**
> Hash functions, chaining, open addressing and the load-factor analysis, all with the proofs
> §1.5 only measured. The division and multiplication methods, and universal hashing — the formal
> answer to §3's adversary — are here. This chapter *is* Part 1, done rigorously.

**2. [Denial of Service via Algorithmic Complexity Attacks](https://www.usenix.org/legacy/events/sec03/tech/full_papers/crosby/crosby.pdf)** —
Crosby & Wallach, USENIX Security 2003. **Free.**
> The paper that named the class. It attacks hash tables and quicksort — the same lesson as §3 and
> NB-02 §3 — and argues that average-case analysis is a *security* liability when inputs are
> adversarial. Twenty years on it reads as a warning that mostly went unheeded until 2011.

**3. [Effective Denial of Service Attacks against Web Application Platforms](https://fahrplan.events.ccc.de/congress/2011/Fahrplan/attachments/2007_28C3_Effective_DoS_on_web_application_platforms.pdf)** —
Klink & Wälde, 28C3, 2011. **Free.**
> The disclosure that forced the fixes in §3.3. It showed PHP, Java, Python, Ruby and others were
> all exploitable with a single POST request. This is *why* `PYTHONHASHSEED` and Java's
> treeification exist — read it to see the defences as responses to a specific, dated event.

## The source behind each section

| Section | Where it comes from | Free? |
|---|---|---|
| 1.1 — uniformity, $\chi^2$ | **CLRS ch. 11.3** on hash functions; **Knuth TAOCP vol. 3 §6.4** | 🔍 |
| 1.2 — birthday bound | **CLRS** exercise 11.3; the birthday problem, any probability text | 🔍 |
| 1.3–1.4 — chaining, probing, tombstones | **CLRS ch. 11.2, 11.4**; **Sedgewick & Wayne, *Algorithms* ch. 3.4** | 🔍 |
| 1.5 — the probing formula $\frac{1}{2}(1+(1-\alpha)^{-2})$ | **Knuth TAOCP vol. 3 §6.4**, Theorem K — the original linear-probing analysis | 🔍 |
| 1.6 — the `hashCode`/`equals` contract | **`Object.hashCode` Javadoc**; **Bloch, *Effective Java* items 10–11** | ✅ (Javadoc) |
| 3 — adversarial input | **Crosby & Wallach 2003**; **Klink & Wälde 2011** | ✅ |
| 3.3 — SipHash | **Aumasson & Bernstein, [*SipHash: a fast short-input PRF*](https://www.aumasson.jp/siphash/siphash.pdf)**, 2012 — the function CPython adopted | ✅ |
| 3.3 — Java treeification | **JDK-8023463**, and the `HashMap` source comment on `TREEIFY_THRESHOLD` | ✅ |
| Q9 — CPython's dict | **`Objects/dictobject.c`** header comment; **Raymond Hettinger's "Modern Dictionaries" talk** | ✅ |
| Ch. 3 — perfect hashing | **Fredman, Komlós & Szemerédi**, *Storing a sparse table with $O(1)$ access*, JACM 1984 (FKS); **CLRS ch. 11.5** | 🔍 |

**Legend:** ✅ free at the link · 🔍 search the exact title on
[Google Scholar](https://scholar.google.com)

### If you read only one

**Crosby & Wallach, 2003.** It is the intellectual center of this notebook: the observation that
"expected $O(1)$" is a statement about a *distribution of inputs*, and that a distribution is not
something you get to assume when someone is choosing your inputs on purpose. §3 is one worked
example of their thesis; NB-02 §3's string-matching attack is another. Once you have read it you
will not look at an average-case bound the same way — you will ask "average over what, chosen by
whom?", which is the right question.

Then read **CLRS ch. 11** for the proofs behind Part 1, and **Klink & Wälde** to watch the theory
become a real outage.

***
# Appendix

| Symptom | Cause | Fix |
|---|---|---|
| Dict build pins a CPU for minutes | Colliding keys → $\Theta(n^2)$ hash flooding (§3.2) | Cap untrusted input size; never `PYTHONHASHSEED=0` in prod |
| `x in collection` loop is slow | `collection` is a `list`: $O(n)$ per test (§2.4) | Make it a `set` or `dict` |
| `list(set(xs))` reorders between runs | Sets have no order; string hashes are randomised (§2.4, §3.3) | `dict.fromkeys(xs)` or an explicit `seen` set |
| Java map has two entries for one key | `equals` overridden, `hashCode` not (§1.6) | Override both; keep `-Xlint:all -Werror` on |
| Key vanishes but `size()` still counts it | Mutated after insertion; hash no longer matches (§1.6) | Immutable keys (`record`, `final`, tuples) |
| `unhashable type` on dict insertion | Mutable key (`list`) or `__eq__` without `__hash__` (§1.6, §2.1) | Use a tuple; define `__hash__` alongside `__eq__` |
| Open-addressed lookups miss after deletes | `delete` cleared the slot instead of tombstoning (§1.4) | Tombstone on delete; rebuild when tombstones pile up |
| Table slows down though it looks half-empty | Tombstones counted as empty (§1.4) | Track used-including-tombstones; resize on that |
| Subarray-sum count off by a few | Missing the `{0: 1}` prefix seed (§2.3) | Seed the map with the empty prefix |
| `{1: 'a', True: 'b'}` has one entry | `True == 1` and hashes equal (Q11) | Do not mix `bool`/`int` (or `int`/`float`) keys |
| Probe counts explode near full | $\alpha$ too high; cost is $(1-\alpha)^{-2}$ (§1.5) | Resize by 0.66–0.75; presize if final size known |
| `%` gives negatives on prefix-mod keys | Language `%` semantics on negatives (§5.5, NB-00) | `((p % k) + k) % k` |

## Checklist for hash-table code

- [ ] Is any `in` test inside a loop hitting a **list** instead of a set/dict (§2.4)?
- [ ] Do the keys' `hashCode`/`__hash__` and `equals`/`__eq__` agree on the same fields (§1.6)?
- [ ] Are the keys **immutable** for as long as they are in the table (§1.6)?
- [ ] Does any dedup rely on `list(set(...))` where order actually matters (§2.4)?
- [ ] Is untrusted input capped before it becomes dictionary keys (§3)?
- [ ] Is `PYTHONHASHSEED` left at its default (never `0`) in production (§3.3)?
- [ ] For a known final size, is the table **presized** to skip resizes (Q5)?
- [ ] If open-addressed: tombstones on delete, and resize on used-not-live (§1.4)?
- [ ] If a custom hash faces untrusted input, is it **keyed** with a random salt (§3, NB-02 §2.4)?
- [ ] Was the complexity **measured**, not assumed (every section)?

## Where to go next

| Notebook | Why it follows |
|---|---|
| `linked_lists_zero_to_hero.ipynb` | §1.3's chains are linked lists; Practice 4's LRU cache welds a map to one |
| `bst_zero_to_hero.ipynb` | The worst-case $O(\log n)$ guarantee §3 and Q10 keep pointing at |
| `tries_zero_to_hero.ipynb` | Prefix and range queries a hash table cannot do (Q10) |
| [`strings_zero_to_hero.ipynb`](strings_zero_to_hero.ipynb) | §2.4's rolling hash and the same adversary, where this notebook's §3 began |

See [`README.md`](README.md) for the full roster and reading order.